# Final Ablation Study — LODO Static + Active Learning Simulation

This notebook contains the complete ablation study:

**LODO (Leave-One-Dataset-Out) static pipeline** trains once on the training pool and scores the held-out test set in a single pass. This gives fast, interpretable baselines across all four ablation conditions (C1–C4).


## 0. Install dependencies

Run this cell once per Colab session. It installs PyTorch Geometric, HuggingFace Transformers, and a few helpers. The `--quiet` flag suppresses the wall of pip output — remove it if something fails and you need to debug.

In [ ]:
# Check whether we're running on Colab so we can skip install if running locally
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # PyTorch Geometric requires knowing your torch + CUDA version at install time.
    # torch.version.cuda returns None on CPU-only Colab, hence the fallback to 'cpu'.
    import torch
    TORCH_VERSION = torch.__version__.split('+')[0]   # e.g. '2.3.0'
    CUDA_VERSION  = torch.version.cuda or 'cpu'       # e.g. '12.1' or 'cpu'
    print(f'PyTorch {TORCH_VERSION} | CUDA {CUDA_VERSION}')

    # Install PyG core + optional sparse ops (needed for efficient GAT)
    !pip install torch_geometric --quiet
    !pip install pyg_lib torch_scatter torch_sparse \
        -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html --quiet

    # HuggingFace ecosystem: transformers for SciBERT, datasets for utilities
    !pip install transformers datasets --quiet

    # asreview-insights gives us the official Normalised Loss implementation
    # so our metric matches exactly what the benchmark paper reports
    !pip install "asreview>=3.0" asreview-insights --quiet

    # networkx for citation graph construction, scipy for TF-IDF sparse ops
    !pip install networkx scikit-learn scipy --quiet

    print('All dependencies installed.')
else:
    print('Running locally — assuming dependencies are already installed.')

## 1. Imports and global configuration

Everything is imported here at the top so it's easy to see what the notebook depends on. The `SEED` constant is used everywhere that randomness appears — sklearn splits, PyTorch weight init, numpy shuffles. Fixing a single seed means you can re-run the notebook and get the same numbers.

In [ ]:
import os
import ast
import warnings
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# sklearn: TF-IDF, LSA (TruncatedSVD), SVM, and metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# PyTorch Geometric: the GAT layer and Data container
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.utils import add_self_loops, to_undirected

# HuggingFace Transformers: SciBERT
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings('ignore')  # suppress sklearn convergence warnings

# Global constants 
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Where your FORAS CSV files live. On Colab, mount Drive and adjust this path.
DATA_DIR = Path('../Week 4 - Ablation Study/data')

# Where SciBERT embeddings get cached after the first run.
# Generating embeddings takes a few minutes per dataset — caching means
# you only pay that cost once, and subsequent runs load from disk instantly.
EMBED_CACHE_DIR = Path('../Week 4 - Ablation Study/cache/scibert_embeddings')
EMBED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Where trained GAT models get cached after the first run.
# Training a GAT takes several minutes per dataset — caching means
# you only pay that cost once, and subsequent runs load from disk instantly.
GAT_CACHE_DIR = Path('../Week 4 - Ablation Study/cache/gat_models')
GAT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# All experiment results go to one CSV. Never scatter results across notebooks.
RESULTS_PATH = Path('results/all_results.csv')
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

# Device: use GPU if available (Colab Pro gives A100 or V100)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'Global seed: {SEED}')

# ASReview — used in Part 2 (Active Learning simulation)
import asreview as asr
from asreview import ActiveLearningCycle, Simulate
from asreview.models.balancers import Balanced
from asreview.models.classifiers import SVM as ASR_SVM
from asreview.models.feature_extractors import Tfidf
from asreview.models.queriers import Max, TopDown
from asreview.models.stoppers import IsFittable
from asreview.models.models import get_ai_config


## 2. Dataset partition

These are the datasets selected from the 120 FORAS/SYNERGY+ CSVs using a 4-criterion scoring system (size, graph density, inclusion rate, giant component fraction, max=10). Full selection process is in `synergy_dataset_visualisation.ipynb`.

The partition was decided in the Week 2 DISC meeting (1 May 2026):
- **RQ1 (10 datasets):** cross-domain representative set — all four conditions run here under LODO evaluation
- **RQ2 Domain A — Osteoarthritis (6 datasets):** within-domain graph density analysis
- **RQ2 Domain B — PTSD/Trauma (6 datasets):** within-domain graph density analysis

Note: Pijls_2018 and Tumkaya_2018 appear in both RQ1 and their respective RQ2 domains. Their LODO folds from RQ1 are reused directly — no extra compute needed.

In [ ]:
# Each entry is: (csv_filename_stem, domain_label, composite_score)
# csv_filename_stem should match the actual file on disk without the .csv extension

RQ1_DATASETS = [
    ('Greca_2023',          'oncology',         10),
    ('Pijls_2018',          'osteoarthritis',   10),
    ('Tumkaya_2018',        'ptsd_trauma',       9),
    ('Rinne_2021',          'ptsd_trauma',       9),
    ('Pinos-Cisneros_2023', 'osteoarthritis',    8),
    ('Abgaz_2023',          'mhealth',           8),
    ('Erer_2015',           'oncology',          8),
    ('Demirkaya_2015',      'neurology',         8),
    ('Leenaars_2020',       'nutrition',         8),
    ('Eggmann_2023',        'oncology',          8),
]

RQ2_DOMAIN_A = [
    ('Pijls_2018',          'osteoarthritis',   10),
    ('Cozim-Melges_2024',   'osteoarthritis',    9),
    ('Pinos-Cisneros_2023', 'osteoarthritis',    8),
    ('van_der_Valk_2021',   'osteoarthritis',    8),
    ('Meijboom_2021',       'osteoarthritis',    8),
    ('Tektonidou_2019',     'osteoarthritis',    8),
]

RQ2_DOMAIN_B = [
    ('Tumkaya_2018',           'ptsd_trauma',    9),
    ('Rinne_2021',             'ptsd_trauma',    9),
    ('Donners_2021',           'ptsd_trauma',    8),
    ('Smid_2019',              'ptsd_trauma',    8),
    ('Boersma-van_Dam_2024',   'ptsd_trauma',    8),
    ('Oud_2018',               'ptsd_trauma',    8),
]

# Convenience: one flat list for iterating all unique datasets
ALL_UNIQUE = list({d[0]: d for d in RQ1_DATASETS + RQ2_DOMAIN_A + RQ2_DOMAIN_B}.values())

print(f'RQ1 pool:      {len(RQ1_DATASETS)} datasets')
print(f'RQ2 Domain A:  {len(RQ2_DOMAIN_A)} datasets (osteoarthritis)')
print(f'RQ2 Domain B:  {len(RQ2_DOMAIN_B)} datasets (PTSD/trauma)')
print(f'Unique total:  {len(ALL_UNIQUE)} datasets')

## 3. Data loading

Each FORAS CSV has the following columns:
- `openalex_id`: unique paper identifier (e.g. `W2345678901`) — **case matters**, see note below
- `label_included`: binary relevance label (1 = relevant, 0 = irrelevant)
- `title`, `abstract`: text fields
- `referenced_works`: stringified Python list of OpenAlex IDs that this paper cites
- `publication_year`, `cited_by_count`: metadata

**Critical case-normalisation note:** The `openalex_id` column stores IDs with a lowercase `w` (e.g. `w2345678901`) but the `referenced_works` lists use uppercase `W` (e.g. `W2345678901`). If you don't normalise, citation matching will silently fail and you'll get an empty graph. Everything is lowercased here.

In [ ]:
def load_dataset(name: str) -> pd.DataFrame:
    """
    Load a single FORAS CSV and apply all necessary cleaning steps.

    Returns a DataFrame with a clean integer index and the columns:
      openalex_id, label_included, title, abstract, text,
      referenced_works (as a Python list), publication_year, cited_by_count
    """
    path = DATA_DIR / f'{name}.csv'
    if not path.exists():
        raise FileNotFoundError(
            f'Expected {path}. Make sure DATA_DIR points to your FORAS folder.'
        )

    df = pd.read_csv(path, low_memory=False)

    # 1. Case-normalise OpenAlex IDs to lowercase 
    # The openalex_id column has lowercase 'w' but referenced_works entries
    # have uppercase 'W'. Normalising both to lowercase makes matching work.
    df['openalex_id'] = df['openalex_id'].str.lower().str.strip()

    # 2. Parse referenced_works from string to Python list 
    # The column is stored as a string like "['W123', 'W456']".
    # ast.literal_eval safely converts it to an actual Python list.
    # We also lowercase every ID inside the list for consistent matching.
    def parse_refs(val):
        if pd.isna(val) or val == '' or val == '[]':
            return []
        try:
            refs = ast.literal_eval(val)
            return [r.lower().strip() for r in refs if isinstance(r, str)]
        except Exception:
            return []

    df['referenced_works'] = df['referenced_works'].apply(parse_refs)

    # 3. Fill missing text fields with empty string 
    # Missing abstracts are common in citation datasets. An empty string won't
    # break TF-IDF or SciBERT tokenisation (they'll just produce a zero vector
    # or a [CLS][SEP] embedding — still valid, just less informative).
    df['title']    = df['title'].fillna('').str.strip()
    df['abstract'] = df['abstract'].fillna('').str.strip()

    # 4. Concatenate title + abstract into one text field 
    # Both TF-IDF and SciBERT will use this single 'text' column.
    # Separating with a period+space so the tokeniser sees a sentence boundary.
    df['text'] = df['title'] + '. ' + df['abstract']
    df['text'] = df['text'].str.strip('. ')  # remove leading/trailing artifacts

    # 5. Ensure label column is integer 
    df['label_included'] = df['label_included'].fillna(0).astype(int)

    # 6. Reset index so iloc and positional indexing stay consistent 
    df = df.reset_index(drop=True)

    return df


def dataset_summary(df: pd.DataFrame, name: str) -> None:
    """Print a one-line summary of a loaded dataset."""
    n_total    = len(df)
    n_relevant = df['label_included'].sum()
    inc_rate   = n_relevant / n_total * 100
    n_with_abs = (df['abstract'].str.len() > 10).sum()
    n_with_refs = (df['referenced_works'].apply(len) > 0).sum()
    print(
        f'  {name:<30} '
        f'{n_total:>5} papers | '
        f'{n_relevant:>4} relevant ({inc_rate:4.1f}%) | '
        f'{n_with_abs:>5} have abstract | '
        f'{n_with_refs:>5} have references'
    )


# Load all unique datasets and run sanity checks
print('Loading datasets...')
print(f'  {"Dataset":<30} {"Papers":>5}   {"Relevant":>9}   {"Has abstract":>12}   {"Has refs":>9}')
print('  ' + '-' * 80)

DATASETS = {}  # name → DataFrame
for (name, domain, score) in ALL_UNIQUE:
    try:
        df = load_dataset(name)
        DATASETS[name] = df
        dataset_summary(df, name)
    except FileNotFoundError as e:
        print(f'  WARNING: {e}')

print(f'\nLoaded {len(DATASETS)} datasets successfully.')

## 4. Normalised Loss metric

Normalised Loss is the primary evaluation metric throughout this thesis. It measures how far the model's recall curve is from the ideal recall curve, normalised so the value is comparable across datasets with different inclusion rates.

**Formula:**
$$\text{NL} = \frac{\text{Optimal AUC} - \text{Actual AUC}}{\text{Optimal AUC} - \text{Worst AUC}}$$

- `0` = the model found all relevant papers first (perfect screening efficiency)
- `1` = the model performed no better than random ordering
- Lower is always better

Unlike WSS@95, Normalised Loss does not depend on an arbitrary recall threshold (95%) and is properly normalised for comparison across datasets with inclusion rates ranging from 0.2% to 23.7%.

In [ ]:
def normalised_loss(labels: np.ndarray, scores: np.ndarray) -> float:
    """
    Compute Normalised Loss from predicted relevance scores and true labels.

    This is a pure-Python implementation that matches the asreview-insights
    calculation exactly (verified against the CLI output in the sanity check).

    Args:
        labels: ground truth binary labels, shape (N,), values in {0, 1}
        scores: predicted relevance probabilities, shape (N,), higher = more relevant

    Returns:
        float in [0, 1]; lower is better
    """
    # Sort papers by descending score — this is the order a screener would see them
    order = np.argsort(scores)[::-1]  # indices from most to least relevant
    sorted_labels = labels[order]     # labels in screening order

    N_total    = len(labels)           # total number of papers
    N_relevant = labels.sum()          # total number of relevant papers

    if N_relevant == 0 or N_relevant == N_total:
        # Degenerate case: no relevant papers or all relevant — loss is undefined.
        # Return 0 (best case) to avoid division by zero breaking the results CSV.
        return 0.0

    # Actual AUC: cumulative sum of found relevant papers at each screening step
    # cumsum[i] = number of relevant papers found after screening i+1 papers
    cumulative_recall = np.cumsum(sorted_labels)
    actual_auc = cumulative_recall.sum()

    # Optimal AUC: all relevant papers come first, then all irrelevant ones
    # This is the area under the best possible recall curve
    optimal_auc = (
        N_total * N_relevant
        - (N_relevant * (N_relevant - 1)) / 2
    )

    # Worst AUC: random ordering — relevant papers are uniformly distributed
    # This is the denominator normalisation term
    worst_auc = N_relevant * (N_relevant + 1) / 2

    loss = (optimal_auc - actual_auc) / (optimal_auc - worst_auc)

    # Clamp to [0, 1] — floating point can produce tiny negative values or > 1
    return float(np.clip(loss, 0.0, 1.0))


def wss_at_95(labels: np.ndarray, scores: np.ndarray) -> float:
    """
    Compute WSS@95 (Work Saved over Sampling at 95% recall).

    Kept here for comparison / sanity checking. NOT the primary metric.
    WSS@95 = fraction of papers not screened when 95% recall is achieved,
    relative to random sampling.

    Returns:
        float in [0, 1]; higher is better (opposite direction from Normalised Loss)
    """
    order         = np.argsort(scores)[::-1]
    sorted_labels = labels[order]
    N_total       = len(labels)
    N_relevant    = labels.sum()

    if N_relevant == 0:
        return 0.0

    # Find the first position where cumulative recall >= 95%
    target = 0.95 * N_relevant
    cumulative = np.cumsum(sorted_labels)
    positions_above = np.where(cumulative >= target)[0]

    if len(positions_above) == 0:
        # Never reached 95% recall — shouldn't happen, but fail gracefully
        return 0.0

    # position is 0-indexed, so position + 1 papers were screened
    n_screened = positions_above[0] + 1
    wss = 1.0 - (n_screened / N_total) - (1 - 0.95)
    return float(np.clip(wss, 0.0, 1.0))


# Quick self-test 
# Perfect ranker: all 5 relevant papers come first
_labels = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
_scores = np.array([0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.0])
assert normalised_loss(_labels, _scores) == 0.0, 'Perfect ranker should give NL=0'

# Worst ranker: all relevant papers come last
_scores_worst = _scores[::-1]
assert normalised_loss(_labels, _scores_worst) == 1.0, 'Worst ranker should give NL=1'

print('Normalised Loss self-test passed.')
print(f'  Perfect ranker: NL = {normalised_loss(_labels, _scores):.4f} (expected 0.0)')
print(f'  Worst ranker:   NL = {normalised_loss(_labels, _scores_worst):.4f} (expected 1.0)')
print(f'  WSS@95 perfect: {wss_at_95(_labels, _scores):.4f} (higher = better)')

## 5. Results logging

A single function that appends one row per experiment run to `results/all_results.csv`. Every number in the thesis comes from this file — never from a notebook variable or a print statement. The `append` pattern means runs accumulate even across separate sessions.

In [ ]:
def log_result(
    dataset:   str,
    held_out:  str,     # which dataset was the test set in this LODO fold
    model:     str,     # condition name, e.g. 'C1_TFIDF_SVM'
    run:       int,     # seed index (0..N_RUNS-1)
    seed:      int,
    nl:        float,   # Normalised Loss — primary metric
    wss95:     float,   # WSS@95 — secondary / sanity check
    runtime_s: float,   # wall-clock seconds for this run
    notes:     str = '',
) -> None:
    """
    Append one result row to the master results CSV.

    The 'held_out' column records which dataset was excluded from training.
    In a LODO setup 'dataset' and 'held_out' are the same — they're both
    here to make the file self-documenting if you later add other eval modes.
    """
    row = pd.DataFrame([{
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'dataset':   dataset,
        'held_out':  held_out,
        'model':     model,
        'run':       run,
        'seed':      seed,
        'nl':        round(nl,    4),
        'wss95':     round(wss95, 4),
        'runtime_s': round(runtime_s, 2),
        'notes':     notes,
    }])

    # Append mode with header written only once
    write_header = not RESULTS_PATH.exists()
    row.to_csv(RESULTS_PATH, mode='a', header=write_header, index=False)


def load_results() -> pd.DataFrame:
    """Load all logged results. Returns empty DataFrame if no results yet."""
    if RESULTS_PATH.exists():
        return pd.read_csv(RESULTS_PATH)
    return pd.DataFrame()


print(f'Results will be logged to: {RESULTS_PATH.resolve()}')

# If there are already results from a previous run, show a summary
existing = load_results()
if len(existing) > 0:
    print(f'Found {len(existing)} existing result rows:')
    print(existing.groupby('model')['nl'].describe().round(4))
else:
    print('No existing results — starting fresh.')

## 6. LODO split utility

Leave-One-Dataset-Out (LODO) is the evaluation strategy agreed with Rens on 1 May. For a pool of N datasets:
- Fold `k`: dataset `k` is the **test set**, all others form the **train+validation set**
- Repeat for each k from 0 to N-1
- Final performance = mean Normalised Loss across all held-out folds

Within the training portion, we do an 80/20 split for train vs validation (for early stopping in the GAT). This split is stratified by label to preserve class ratios in both halves.

In [ ]:
from sklearn.model_selection import train_test_split


def lodo_splits(dataset_names: list[str]) -> list[dict]:
    """
    Generate all LODO folds for a list of dataset names.

    Returns a list of dicts, one per fold:
        {
          'test':  str  — name of the held-out (test) dataset
          'train': list[str] — names of the training datasets
        }
    """
    folds = []
    for i, test_name in enumerate(dataset_names):
        train_names = [n for j, n in enumerate(dataset_names) if j != i]
        folds.append({'test': test_name, 'train': train_names})
    return folds


def make_train_val_arrays(
    train_names: list[str],
    feature_fn,       # callable: df → np.ndarray of shape (N, D)
    val_frac: float = 0.2,
) -> tuple:
    """
    Given a list of training dataset names and a feature extraction function,
    concatenate all training data and split into train/val.

    Args:
        train_names: dataset names that form the training pool
        feature_fn:  function that takes a DataFrame and returns a feature matrix
        val_frac:    fraction of training data to hold out for validation

    Returns:
        X_train, X_val, y_train, y_val as numpy arrays
    """
    X_parts, y_parts = [], []

    for name in train_names:
        df = DATASETS[name]
        X  = feature_fn(df)               # (N_i, D) feature matrix
        y  = df['label_included'].values  # (N_i,) binary labels
        X_parts.append(X)
        y_parts.append(y)

    # Stack all training datasets into one big matrix
    X_all = np.vstack(X_parts)     # (sum of N_i, D)
    y_all = np.concatenate(y_parts)  # (sum of N_i,)

    # Stratified split preserves the overall inclusion rate in both halves
    X_train, X_val, y_train, y_val = train_test_split(
        X_all, y_all,
        test_size=val_frac,
        stratify=y_all,
        random_state=SEED,
    )
    return X_train, X_val, y_train, y_val


# Show what the LODO folds look like for the RQ1 pool
rq1_names  = [d[0] for d in RQ1_DATASETS]
rq1_folds  = lodo_splits(rq1_names)

print('LODO folds for RQ1 pool:')
for i, fold in enumerate(rq1_folds):
    print(f'  Fold {i+1:2d}: test = {fold["test"]:<30}  train = {len(fold["train"])} datasets')

## 7. Text preprocessing (shared across all conditions)

The preprocessing function is defined once and used by both TF-IDF (C1) and SciBERT (C2–C4). The only difference is the `mode` flag that switches between returning raw text strings (for TF-IDF, which does its own tokenisation) and tokenised tensors (for SciBERT).

In [ ]:
def get_texts(df: pd.DataFrame) -> list[str]:
    """
    Return the list of text strings for a dataset.
    The 'text' column was already built in load_dataset() as title + '. ' + abstract.
    This function is a thin wrapper so callers don't need to know the column name.
    """
    return df['text'].tolist()


def get_labels(df: pd.DataFrame) -> np.ndarray:
    """Return the label array as a numpy int array."""
    return df['label_included'].values.astype(int)


# Verify on one dataset before continuing
if DATASETS:
    sample_name = list(DATASETS.keys())[0]
    sample_df   = DATASETS[sample_name]
    sample_texts  = get_texts(sample_df)
    sample_labels = get_labels(sample_df)

    print(f'Sample dataset: {sample_name}')
    print(f'  Total papers:  {len(sample_texts)}')
    print(f'  Relevant:      {sample_labels.sum()} ({sample_labels.mean()*100:.1f}%)')
    print(f'  Example text:  "{sample_texts[0][:120]}..."')

---
## CONDITION 1 — TF-IDF + SVM (Baseline)

This is the static approximation of ASReview's **ELAS-u4** model — the current production
default in ASReview ≥ 3.0. ELAS-u4 uses TF-IDF (bigrams, `ngram_range=(1,2)`) combined with
an SVM classifier in an active learning loop, retraining after every human label.

Here we collapse the active-learning loop into a single train/score pass so C1 fits cleanly
within the LODO evaluation framework: TF-IDF is fitted on the training pool, then LinearSVC
is trained once and scores all held-out test papers in a single forward pass.

The TF-IDF hyperparameters (`ngram_range`, `sublinear_tf`, etc.) and SVM hyperparameters
are loaded directly from `get_ai_config("elas_u4")` to guarantee they match the official
ELAS-u4 definition exactly. The one adaptation is using `LinearSVC.decision_function` for
ranking instead of ASReview's internal `predict_proba` wrapper — this is equivalent for
ranking purposes and avoids the asreview `Simulate` overhead in a static pipeline.

> **Note:** ELAS-u3 (the previous default) used TF-IDF + Naive Bayes. ELAS-u4 replaced NB
> with SVM, which gives better calibrated relevance scores on imbalanced corpora.


In [ ]:
import time
from asreview.models.models import get_ai_config

# Load official ELAS-u4 hyperparameters directly from the asreview package
# so they are guaranteed to stay in sync with future package updates.
_elas_u4_config = get_ai_config("elas_u4").get("value")
_tfidf_params   = _elas_u4_config.feature_extractor_param   # e.g. sublinear_tf, ngram_range
_svm_params     = _elas_u4_config.classifier_param           # e.g. C, class_weight

print(f"ELAS-u4 TF-IDF params : {_tfidf_params}")
print(f"ELAS-u4 SVM params    : {_svm_params}")


def build_tfidf_features(texts: list[str]) -> TfidfVectorizer:
    """
    Fit a TF-IDF vectoriser using the official ELAS-u4 hyperparameters.

    Key settings (loaded from get_ai_config):
    - sublinear_tf=True : log(1 + tf) dampens very frequent terms
    - ngram_range=(1,2) : unigrams + bigrams (ELAS-u4 default; ELAS-u3 used (1,1))
    - stop_words=None   : no stop word removal (matching ASReview's default)
    """
    vectoriser = TfidfVectorizer(**_tfidf_params)
    vectoriser.fit(texts)
    return vectoriser


def run_condition1_lodo(
    dataset_names: list[str],
    n_runs: int = 5,
) -> list[dict]:
    """
    Run Condition 1 (ELAS-u4: TF-IDF bigrams + LinearSVC) under LODO evaluation.

    For each fold:
      1. Fit TF-IDF on all training texts (ELAS-u4 params, including bigrams)
      2. Transform both training and test texts
      3. Train LinearSVC with ELAS-u4 SVM hyperparameters
      4. Score test papers using decision_function (higher = more relevant)
      5. Compute Normalised Loss

    Args:
        dataset_names: ordered list of dataset names forming the pool
        n_runs:        number of times to repeat each fold (different seeds)
                       LinearSVC is deterministic, so repeated runs test
                       sensitivity to prior seeding only.

    Returns:
        List of result dicts (also logged to CSV)
    """
    folds   = lodo_splits(dataset_names)
    results = []

    for fold in folds:
        test_name   = fold['test']
        train_names = fold['train']

        test_df     = DATASETS[test_name]
        test_texts  = get_texts(test_df)
        test_labels = get_labels(test_df)

        # Gather all training texts from the training pool
        train_texts  = [t for n in train_names for t in get_texts(DATASETS[n])]
        train_labels = np.concatenate([get_labels(DATASETS[n]) for n in train_names])

        # Fit TF-IDF on training texts only — test data never seen during fit
        vectoriser = build_tfidf_features(train_texts)

        X_train = vectoriser.transform(train_texts)  # sparse matrix
        X_test  = vectoriser.transform(test_texts)   # same vocabulary

        for run_idx in range(n_runs):
            run_seed = SEED + run_idx
            t0 = time.time()

            # Use ELAS-u4 SVM hyperparameters loaded from get_ai_config.
            # LinearSVC is the sklearn equivalent of ASReview's SVM wrapper;
            # decision_function is used for ranking (matches ASReview's scoring).
            clf = LinearSVC(
                **_svm_params,
                max_iter=2000,
                random_state=run_seed,
            )
            clf.fit(X_train, train_labels)

            # Higher decision_function score = closer to the positive hyperplane
            # = more likely relevant. Used directly as the ranking signal.
            scores = clf.decision_function(X_test)

            nl    = normalised_loss(test_labels, scores)
            wss95 = wss_at_95(test_labels, scores)
            rt    = time.time() - t0

            log_result(
                dataset=test_name, held_out=test_name,
                model='C1_TFIDF_SVM',
                run=run_idx, seed=run_seed,
                nl=nl, wss95=wss95, runtime_s=rt,
            )
            results.append({'dataset': test_name, 'run': run_idx, 'nl': nl})
            print(f'  C1 | fold: {test_name:<28} run {run_idx} | NL={nl:.4f} WSS95={wss95:.4f} ({rt:.1f}s)')

    return results


print('Condition 1 (ELAS-u4: TF-IDF bigrams + SVM) ready. Run the next cell to execute.')


In [ ]:
# Execute Condition 1 on RQ1 pool
# Set N_RUNS to 5 for the thesis. For a quick sanity check, use N_RUNS=1.
N_RUNS = 5

print(f'Running Condition 1 (TF-IDF + SVM) on {len(rq1_names)} datasets × {N_RUNS} runs...')
c1_results = run_condition1_lodo(rq1_names, n_runs=N_RUNS)

In [ ]:
# Summarise
c1_df = pd.DataFrame(c1_results)
print('\nCondition 1 — Mean Normalised Loss per dataset (across runs):')
print(c1_df.groupby('dataset')['nl'].mean().sort_values().to_string())

In [ ]:
# C1 Results Visualisation 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Aggregate across runs
c1_agg = (
    c1_df.groupby('dataset')['nl']
    .agg(mean='mean', std='std', n='count')
    .reset_index()
    .sort_values('mean')
)

# Domain colour map — matches your RQ1 partition
DOMAIN_COLOURS = {
    'Greca_2023':           '#E05A2B',   # oncology
    'Erer_2015':            '#E05A2B',
    'Eggmann_2023':         '#E05A2B',
    'Pijls_2018':           '#534AB7',   # osteoarthritis
    'Pinos-Cisneros_2023':  '#534AB7',
    'Tumkaya_2018':         '#1D9E75',   # ptsd/trauma
    'Rinne_2021':           '#1D9E75',
    'Abgaz_2023':           '#C0A020',   # mhealth
    'Demirkaya_2015':       '#888888',   # neurology
    'Leenaars_2020':        '#3399CC',   # nutrition
}

colours = [DOMAIN_COLOURS.get(d, '#AAAAAA') for d in c1_agg['dataset']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Condition 1 — TF-IDF + SVM (LODO)', fontsize=13, fontweight='bold', y=1.01)

# Left: mean NL per dataset with error bars 
ax = axes[0]
x = np.arange(len(c1_agg))

bars = ax.barh(x, c1_agg['mean'], xerr=c1_agg['std'],
               color=colours, alpha=0.85, capsize=4,
               error_kw={'linewidth': 1.0, 'ecolor': '#333333'})

# Reference lines
ax.axvline(0.5, color='#999999', linestyle='--', linewidth=1, label='Random baseline (0.5)')
ax.axvline(c1_agg['mean'].mean(), color='#CC2222', linestyle='-',
           linewidth=1.5, label=f'Pool mean ({c1_agg["mean"].mean():.3f})')

ax.set_yticks(x)
ax.set_yticklabels(c1_agg['dataset'], fontsize=9)
ax.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax.set_title('Mean NL per dataset  ±1 SD across runs', fontsize=10)
ax.set_xlim(0, max(0.7, c1_agg['mean'].max() + c1_agg['std'].max() + 0.05))
ax.legend(fontsize=8, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)

# Annotate each bar with its mean value
for i, row in enumerate(c1_agg.itertuples()):
    ax.text(row.mean + row.std + 0.01, i, f'{row.mean:.3f}',
            va='center', fontsize=8, color='#333333')

# Right: run-level distribution (strip + box) 
ax2 = axes[1]

datasets_sorted = c1_agg['dataset'].tolist()
for i, ds in enumerate(datasets_sorted):
    vals = c1_df[c1_df['dataset'] == ds]['nl'].values
    # jitter x for strip plot
    jitter = np.random.uniform(-0.15, 0.15, size=len(vals))
    ax2.scatter(vals, np.full_like(vals, i) + jitter,
                color=DOMAIN_COLOURS.get(ds, '#AAAAAA'),
                alpha=0.7, s=40, zorder=3)
    # median line
    ax2.plot([np.median(vals), np.median(vals)], [i - 0.3, i + 0.3],
             color='#222222', linewidth=1.5, zorder=4)

ax2.axvline(0.5, color='#999999', linestyle='--', linewidth=1)
ax2.set_yticks(range(len(datasets_sorted)))
ax2.set_yticklabels(datasets_sorted, fontsize=9)
ax2.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax2.set_title(f'Run-level spread  (n={int(c1_agg["n"].iloc[0])} runs per dataset)', fontsize=10)
ax2.set_xlim(0, max(0.7, c1_df['nl'].max() + 0.05))
ax2.spines[['top', 'right']].set_visible(False)

# Domain legend
legend_patches = [
    mpatches.Patch(color='#E05A2B', label='Oncology'),
    mpatches.Patch(color='#534AB7', label='Osteoarthritis'),
    mpatches.Patch(color='#1D9E75', label='PTSD/Trauma'),
    mpatches.Patch(color='#C0A020', label='mHealth'),
    mpatches.Patch(color='#888888', label='Neurology'),
    mpatches.Patch(color='#3399CC', label='Nutrition'),
]
ax2.legend(handles=legend_patches, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('results/c1_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Console summary 
print('\nC1 — Normalised Loss summary')
print(f'  Pool mean ± std : {c1_agg["mean"].mean():.4f} ± {c1_agg["mean"].std():.4f}')
print(f'  Best dataset    : {c1_agg.iloc[0]["dataset"]}  ({c1_agg.iloc[0]["mean"]:.4f})')
print(f'  Worst dataset   : {c1_agg.iloc[-1]["dataset"]}  ({c1_agg.iloc[-1]["mean"]:.4f})')
print(f'\n  Per-dataset breakdown:')
for row in c1_agg.itertuples():
    bar = '█' * int(row.mean * 30)
    print(f'  {row.dataset:<28}  {row.mean:.4f} ±{row.std:.4f}  {bar}')

---
## CONDITION 2a — SciBERT-only

SciBERT (`allenai/scibert_scivocab_uncased`) is a BERT model pretrained on 1.14 million scientific papers from Semantic Scholar. Compared to vanilla BERT, its vocabulary is built from scientific text — words like "osteoarthritis", "PTSD", and "metastatic" are actual tokens rather than split into subwords.

We use SciBERT as a **frozen feature extractor**: the weights are never updated. We pass each paper's text through the encoder, take the `[CLS]` token's hidden state as a 768-dimensional summary vector, and feed that into a logistic regression classifier. Frozen inference is fast enough on Colab Pro CPU for ~20 datasets.

In [ ]:
SCIBERT_MODEL = 'allenai/scibert_scivocab_uncased'
SCIBERT_MAXLEN = 512   # SciBERT's maximum sequence length
EMBED_BATCH = 32       # papers processed per forward pass; reduce if OOM


def load_scibert():
    """
    Load the SciBERT tokeniser and encoder. Called once and reused.

    Returns:
        tokenizer: HuggingFace fast tokenizer for SciBERT
        model: SciBERT encoder, eval mode, on DEVICE
    """
    print(f'Loading SciBERT from HuggingFace Hub ({SCIBERT_MODEL})...')
    tokenizer = AutoTokenizer.from_pretrained(SCIBERT_MODEL)
    model     = AutoModel.from_pretrained(SCIBERT_MODEL)

    # eval() disables dropout and batch norm updates — critical for feature
    # extraction. Without this, repeated calls on the same text would give
    # slightly different embeddings (dropout is random at train time).
    model.eval()
    model.to(DEVICE)

    # Freeze all parameters — nothing should be updated during our pipeline
    for param in model.parameters():
        param.requires_grad = False

    print(f'SciBERT loaded. Parameters: {sum(p.numel() for p in model.parameters()):,} (all frozen)')
    return tokenizer, model


@torch.no_grad()  # disables gradient computation — saves ~3x memory
def embed_texts(
    texts: list[str],
    tokenizer,
    model,
    batch_size: int = EMBED_BATCH,
) -> np.ndarray:
    """
    Generate SciBERT [CLS] embeddings for a list of texts.

    The [CLS] token is the first token in every BERT sequence. After training,
    its hidden state is a compressed representation of the entire input — the
    model was designed to use it for classification tasks.

    Processing in batches avoids loading all papers into GPU memory at once.

    Returns:
        np.ndarray of shape (len(texts), 768)
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]

        # Tokenise: pad short sequences, truncate long ones to 512 tokens
        encoded = tokenizer(
            batch_texts,
            padding=True,        # pad to the longest sequence in this batch
            truncation=True,     # cut to SCIBERT_MAXLEN if longer
            max_length=SCIBERT_MAXLEN,
            return_tensors='pt', # return PyTorch tensors
        )

        # Move input tensors to the same device as the model
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        # Forward pass through SciBERT
        # output.last_hidden_state: (batch_size, sequence_len, 768)
        output = model(**encoded)

        # Take the [CLS] token — index 0 in the sequence dimension
        # Shape: (batch_size, 768)
        cls_embeddings = output.last_hidden_state[:, 0, :]

        # Move back to CPU and convert to numpy for sklearn compatibility
        all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)  # (total_texts, 768)


def get_scibert_embeddings(
    dataset_name: str,
    df: pd.DataFrame,
    tokenizer,
    model,
) -> np.ndarray:
    """
    Get SciBERT embeddings for a dataset, using the disk cache if available.

    The cache saves compute time: generating embeddings for a dataset with
    20,000 papers takes ~5 minutes on Colab Pro. After the first run, loading
    from .npy takes ~0.1 seconds.

    Returns:
        np.ndarray of shape (N_papers, 768)
    """
    cache_path = EMBED_CACHE_DIR / f'{dataset_name}_scibert.npy'

    if cache_path.exists():
        embs = np.load(cache_path)
        print(f'  Loaded cached embeddings for {dataset_name}: shape {embs.shape}')
        return embs

    print(f'  Computing SciBERT embeddings for {dataset_name} ({len(df)} papers)...')
    texts = get_texts(df)
    embs  = embed_texts(texts, tokenizer, model)
    np.save(cache_path, embs)
    print(f'  Saved to {cache_path} — shape {embs.shape}')
    return embs


print('SciBERT utilities defined. Load the model in the next cell.')

In [ ]:
# Load SciBERT once — reused across Conditions 2 and 4
scibert_tokenizer, scibert_model = load_scibert()

In [ ]:
# Pre-compute and cache SciBERT embeddings for all unique datasets
# This is done upfront so Conditions 2 and 4 both benefit from the cache.

SCIBERT_EMBEDDINGS = {}  # dataset_name → np.ndarray (N, 768)

print('Computing / loading SciBERT embeddings for all datasets...')
for name in DATASETS:
    SCIBERT_EMBEDDINGS[name] = get_scibert_embeddings(
        name, DATASETS[name], scibert_tokenizer, scibert_model
    )

print(f'\nAll embeddings ready. Total cached: {len(SCIBERT_EMBEDDINGS)}')

In [ ]:
from sklearn.linear_model import LogisticRegression


def run_condition2_lodo(
    dataset_names: list[str],
    n_runs: int = 5,
) -> list[dict]:
    """
    Run Condition 2 (SciBERT frozen embeddings + Logistic Regression) under LODO.

    We use Logistic Regression rather than LinearSVC here because:
    - predict_proba() gives well-calibrated probabilities for ranking
    - It's numerically stable on 768-d dense features
    - class_weight='balanced' handles the low inclusion rates
    - max_iter=2000 prevents ConvergenceWarning on large training pools
      (768-d dense features with lbfgs converge quickly in practice, so
       this upper bound is rarely reached)
    """
    folds   = lodo_splits(dataset_names)
    results = []

    for fold in folds:
        test_name   = fold['test']
        train_names = fold['train']

        # Retrieve pre-computed embeddings — no SciBERT forward pass here
        X_test  = SCIBERT_EMBEDDINGS[test_name]
        y_test  = get_labels(DATASETS[test_name])

        X_train = np.vstack([SCIBERT_EMBEDDINGS[n] for n in train_names])
        y_train = np.concatenate([get_labels(DATASETS[n]) for n in train_names])

        # L2 normalise embeddings — cosine similarity becomes dot product
        # after normalisation, which helps logistic regression converge faster
        from sklearn.preprocessing import normalize
        X_train = normalize(X_train, norm='l2')
        X_test  = normalize(X_test,  norm='l2')

        for run_idx in range(n_runs):
            run_seed = SEED + run_idx
            t0 = time.time()

            clf = LogisticRegression(
                class_weight='balanced',
                max_iter=2000,
                random_state=run_seed,
                C=1.0,
                solver='lbfgs',
            )
            clf.fit(X_train, y_train)

            # predict_proba returns [P(class=0), P(class=1)] — we want P(class=1)
            scores = clf.predict_proba(X_test)[:, 1]

            nl    = normalised_loss(y_test, scores)
            wss95 = wss_at_95(y_test, scores)
            rt    = time.time() - t0

            log_result(
                dataset=test_name, held_out=test_name,
                model='C2a_SciBERT_LR',
                run=run_idx, seed=run_seed,
                nl=nl, wss95=wss95, runtime_s=rt,
                notes='frozen_scibert L2norm solver=lbfgs C=1.0',
            )
            results.append({'dataset': test_name, 'run': run_idx, 'nl': nl})
            print(f'  C2a | fold: {test_name:<28} run {run_idx} | NL={nl:.4f} WSS95={wss95:.4f} ({rt:.1f}s)')

    return results


print('Condition 2 (SciBERT-only) ready.')

In [ ]:
print(f'Running Condition 2a (SciBERT + LR) on {len(rq1_names)} datasets × {N_RUNS} runs...')
c2_results = run_condition2_lodo(rq1_names, n_runs=N_RUNS)

In [ ]:
c2_df = pd.DataFrame(c2_results)
print('\nCondition 2a — Mean NL per dataset:')
print(c2_df.groupby('dataset')['nl'].mean().sort_values().to_string())

In [ ]:
# C2a Results Visualisation 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

c2a_agg = (
    c2_df.groupby('dataset')['nl']
    .agg(mean='mean', std='std', n='count')
    .reset_index()
    .sort_values('mean')
)

DOMAIN_COLOURS = {
    'Greca_2023':           '#E05A2B',
    'Erer_2015':            '#E05A2B',
    'Eggmann_2023':         '#E05A2B',
    'Pijls_2018':           '#534AB7',
    'Pinos-Cisneros_2023':  '#534AB7',
    'Tumkaya_2018':         '#1D9E75',
    'Rinne_2021':           '#1D9E75',
    'Abgaz_2023':           '#C0A020',
    'Demirkaya_2015':       '#888888',
    'Leenaars_2020':        '#3399CC',
}

colours = [DOMAIN_COLOURS.get(d, '#AAAAAA') for d in c2a_agg['dataset']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Condition 2a — SciBERT + Logistic Regression (LODO)',
             fontsize=13, fontweight='bold', y=1.01)

# Left: mean NL per dataset 
ax = axes[0]
x = np.arange(len(c2a_agg))

ax.barh(x, c2a_agg['mean'], xerr=c2a_agg['std'],
        color=colours, alpha=0.85, capsize=4,
        error_kw={'linewidth': 1.0, 'ecolor': '#333333'})

# C1 comparison dots — requires c1_agg to be in scope from the C1 visualisation cell
try:
    for i, ds in enumerate(c2a_agg['dataset']):
        c1_val = c1_agg.loc[c1_agg['dataset'] == ds, 'mean'].values
        if len(c1_val):
            ax.scatter(c1_val[0], i, marker='D', s=35,
                       color='#CC2222', zorder=5, label='C1 mean' if i == 0 else '')
    ax.legend(fontsize=8, loc='lower right')
except NameError:
    pass  # c1_agg not in scope — run C1 visualisation cell first for comparison dots

ax.axvline(0.5, color='#999999', linestyle='--', linewidth=1, label='Random (0.5)')
ax.axvline(c2a_agg['mean'].mean(), color='#9FE1CB', linestyle='-',
           linewidth=1.5, label=f'C2a mean ({c2a_agg["mean"].mean():.3f})')

ax.set_yticks(x)
ax.set_yticklabels(c2a_agg['dataset'], fontsize=9)
ax.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax.set_title('Mean NL per dataset  ±1 SD across runs', fontsize=10)
ax.set_xlim(0, max(0.7, c2a_agg['mean'].max() + c2a_agg['std'].max() + 0.05))
ax.spines[['top', 'right']].set_visible(False)

for i, row in enumerate(c2a_agg.itertuples()):
    ax.text(row.mean + row.std + 0.01, i, f'{row.mean:.3f}',
            va='center', fontsize=8, color='#333333')

# Right: run-level strip plot
ax2 = axes[1]

datasets_sorted = c2a_agg['dataset'].tolist()
for i, ds in enumerate(datasets_sorted):
    vals = c2_df[c2_df['dataset'] == ds]['nl'].values
    jitter = np.random.uniform(-0.15, 0.15, size=len(vals))
    ax2.scatter(vals, np.full_like(vals, i) + jitter,
                color=DOMAIN_COLOURS.get(ds, '#AAAAAA'),
                alpha=0.7, s=40, zorder=3)
    ax2.plot([np.median(vals), np.median(vals)], [i - 0.3, i + 0.3],
             color='#222222', linewidth=1.5, zorder=4)

ax2.axvline(0.5, color='#999999', linestyle='--', linewidth=1)
ax2.set_yticks(range(len(datasets_sorted)))
ax2.set_yticklabels(datasets_sorted, fontsize=9)
ax2.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax2.set_title(f'Run-level spread  (n={int(c2a_agg["n"].iloc[0])} runs per dataset)', fontsize=10)
ax2.set_xlim(0, max(0.7, c2_df['nl'].max() + 0.05))
ax2.spines[['top', 'right']].set_visible(False)

legend_patches = [
    mpatches.Patch(color='#E05A2B', label='Oncology'),
    mpatches.Patch(color='#534AB7', label='Osteoarthritis'),
    mpatches.Patch(color='#1D9E75', label='PTSD/Trauma'),
    mpatches.Patch(color='#C0A020', label='mHealth'),
    mpatches.Patch(color='#888888', label='Neurology'),
    mpatches.Patch(color='#3399CC', label='Nutrition'),
]
ax2.legend(handles=legend_patches, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('results/c2a_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Console summary 
print('\nC2a — Normalised Loss summary')
print(f'  Pool mean ± std : {c2a_agg["mean"].mean():.4f} ± {c2a_agg["mean"].std():.4f}')
print(f'  Best dataset    : {c2a_agg.iloc[0]["dataset"]}  ({c2a_agg.iloc[0]["mean"]:.4f})')
print(f'  Worst dataset   : {c2a_agg.iloc[-1]["dataset"]}  ({c2a_agg.iloc[-1]["mean"]:.4f})')

# C1 vs C2a delta — requires c1_agg in scope
try:
    merged = c2a_agg.merge(c1_agg[['dataset', 'mean']], on='dataset', suffixes=('_c2a', '_c1'))
    merged['delta'] = merged['mean_c1'] - merged['mean_c2a']  # positive = C2a better
    print(f'\n  C1 → C2a improvement (positive = C2a better):')
    for row in merged.sort_values('delta', ascending=False).itertuples():
        arrow = '▲' if row.delta > 0 else '▼'
        print(f'  {row.dataset:<28}  {arrow} {abs(row.delta):.4f}')
    print(f'\n  Mean delta: {merged["delta"].mean():+.4f}  '
          f'({"C2a better" if merged["delta"].mean() > 0 else "C1 better"} overall)')
except NameError:
    print('\n  (Run C1 visualisation cell first to see C1 vs C2a comparison)')

In [ ]:
def run_condition2b_lodo(
    dataset_names: list[str],
    n_runs: int = 5,
) -> list[dict]:
    """
    Run Condition 2b (SciBERT frozen embeddings + LinearSVC) under LODO.

    This is the feature isolation control sub-condition. The only difference
    from C2a is the classifier: LinearSVC here vs Logistic Regression in C2a.
    The classifier is identical to C1 (TF-IDF + SVM), so the C1 → C2b gap
    reflects SciBERT feature quality in isolation — no classifier confound.

    The embeddings are already in SCIBERT_EMBEDDINGS from the C2a step.
    This function reuses them directly — no SciBERT forward passes happen here.
    """
    from sklearn.preprocessing import normalize

    folds   = lodo_splits(dataset_names)
    results = []

    for fold in folds:
        test_name   = fold['test']
        train_names = fold['train']

        # Pull pre-computed embeddings — same arrays used in C2a
        X_test  = SCIBERT_EMBEDDINGS[test_name]
        y_test  = get_labels(DATASETS[test_name])

        X_train = np.vstack([SCIBERT_EMBEDDINGS[n] for n in train_names])
        y_train = np.concatenate([get_labels(DATASETS[n]) for n in train_names])

        # L2 normalise — same preprocessing as C2a so the only variable
        # between C2a and C2b is the classifier, nothing else
        X_train = normalize(X_train, norm='l2')
        X_test  = normalize(X_test,  norm='l2')

        for run_idx in range(n_runs):
            run_seed = SEED + run_idx
            t0 = time.time()

            # LinearSVC with identical hyperparameters to C1.
            # decision_function gives a signed distance to the hyperplane —
            # higher = more relevant, used directly as the ranking score.
            clf = LinearSVC(
                class_weight='balanced',
                max_iter=2000,
                random_state=run_seed,
                C=1.0,
            )
            clf.fit(X_train, y_train)
            scores = clf.decision_function(X_test)

            nl    = normalised_loss(y_test, scores)
            wss95 = wss_at_95(y_test, scores)
            rt    = time.time() - t0

            log_result(
                dataset=test_name, held_out=test_name,
                model='C2b_SciBERT_SVM',
                run=run_idx, seed=run_seed,
                nl=nl, wss95=wss95, runtime_s=rt,
                notes='isolation_control: features=SciBERT, classifier=LinearSVC_same_as_C1',
            )
            results.append({'dataset': test_name, 'run': run_idx, 'nl': nl})
            print(f'  C2b | fold: {test_name:<28} run {run_idx} | NL={nl:.4f} WSS95={wss95:.4f} ({rt:.1f}s)')

    return results


# Execute Condition 2b on RQ1 pool 
print(f'Running Condition 2b (SciBERT + SVM isolation control) on {len(rq1_names)} datasets × {N_RUNS} runs...')
c2b_results = run_condition2b_lodo(rq1_names, n_runs=N_RUNS)

c2b_df = pd.DataFrame(c2b_results)
print('\nCondition 2b — Mean NL per dataset:')
print(c2b_df.groupby('dataset')['nl'].mean().sort_values().to_string())



In [ ]:
# Quick decomposition table — shows the three-way split at a glance
print('\n Decomposition: what drove C1 → C2a? ')
print(f'  {"Dataset":<30} {"C1":>8} {"C2b":>8} {"C2a":>8} {"feat Δ":>8} {"clf Δ":>8}')
print('  ' + '-' * 72)

existing = load_results()
for ds in rq1_names:
    c1_nl  = existing[(existing.dataset==ds) & (existing.model=='C1_TFIDF_SVM')]['nl'].mean()
    c2b_nl = existing[(existing.dataset==ds) & (existing.model=='C2b_SciBERT_SVM')]['nl'].mean()
    c2a_nl = existing[(existing.dataset==ds) & (existing.model=='C2a_SciBERT_LR')]['nl'].mean()

    if any(np.isnan([c1_nl, c2b_nl, c2a_nl])):
        continue

    feat_delta = c1_nl  - c2b_nl  # positive = SciBERT features helped vs TF-IDF
    clf_delta  = c2b_nl - c2a_nl  # positive = LR helped vs SVM on SciBERT features

    print(f'  {ds:<30} {c1_nl:>8.4f} {c2b_nl:>8.4f} {c2a_nl:>8.4f} {feat_delta:>+8.4f} {clf_delta:>+8.4f}')

print('\n  feat Δ = C1 - C2b  (positive = SciBERT features beat TF-IDF, classifier held constant)')
print('  clf Δ  = C2b - C2a (positive = LR beats SVM on SciBERT features)')

In [ ]:
# C2b Results Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

c2b_agg = (
    c2b_df.groupby('dataset')['nl']
    .agg(mean='mean', std='std', n='count')
    .reset_index()
    .sort_values('mean')
)

DOMAIN_COLOURS = {
    'Greca_2023':           '#E05A2B',
    'Erer_2015':            '#E05A2B',
    'Eggmann_2023':         '#E05A2B',
    'Pijls_2018':           '#534AB7',
    'Pinos-Cisneros_2023':  '#534AB7',
    'Tumkaya_2018':         '#1D9E75',
    'Rinne_2021':           '#1D9E75',
    'Abgaz_2023':           '#C0A020',
    'Demirkaya_2015':       '#888888',
    'Leenaars_2020':        '#3399CC',
}

colours = [DOMAIN_COLOURS.get(d, '#AAAAAA') for d in c2b_agg['dataset']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Condition 2b — SciBERT + SVM  [feature isolation control] (LODO)',
             fontsize=13, fontweight='bold', y=1.01)

# Left: mean NL per dataset 
ax = axes[0]
x = np.arange(len(c2b_agg))

ax.barh(x, c2b_agg['mean'], xerr=c2b_agg['std'],
        color=colours, alpha=0.85, capsize=4,
        error_kw={'linewidth': 1.0, 'ecolor': '#333333'})

# C1 and C2a overlays for the causal chain read-off
for label, agg_df, marker, colour in [
    ('C1 (TF-IDF+SVM)',   'c1_agg',  'D', '#CC2222'),
    ('C2a (SciBERT+LR)',  'c2a_agg', 's', '#1D9E75'),
]:
    try:
        ref = eval(agg_df)
        for i, ds in enumerate(c2b_agg['dataset']):
            val = ref.loc[ref['dataset'] == ds, 'mean'].values
            if len(val):
                ax.scatter(val[0], i, marker=marker, s=40,
                           color=colour, zorder=5,
                           label=label if i == 0 else '')
    except NameError:
        pass

ax.axvline(0.5, color='#999999', linestyle='--', linewidth=1)
ax.axvline(c2b_agg['mean'].mean(), color='#9FE1CB', linestyle='-',
           linewidth=1.5, label=f'C2b mean ({c2b_agg["mean"].mean():.3f})')

ax.set_yticks(x)
ax.set_yticklabels(c2b_agg['dataset'], fontsize=9)
ax.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax.set_title('Mean NL per dataset  ±1 SD across runs\n'
             '◆ = C1  ■ = C2a  (for causal chain comparison)', fontsize=9)
ax.set_xlim(0, max(0.7, c2b_agg['mean'].max() + c2b_agg['std'].max() + 0.05))
ax.legend(fontsize=8, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)

for i, row in enumerate(c2b_agg.itertuples()):
    ax.text(row.mean + row.std + 0.01, i, f'{row.mean:.3f}',
            va='center', fontsize=8, color='#333333')

# Right: run-level strip plot 
ax2 = axes[1]

datasets_sorted = c2b_agg['dataset'].tolist()
for i, ds in enumerate(datasets_sorted):
    vals = c2b_df[c2b_df['dataset'] == ds]['nl'].values
    jitter = np.random.uniform(-0.15, 0.15, size=len(vals))
    ax2.scatter(vals, np.full_like(vals, i) + jitter,
                color=DOMAIN_COLOURS.get(ds, '#AAAAAA'),
                alpha=0.7, s=40, zorder=3)
    ax2.plot([np.median(vals), np.median(vals)], [i - 0.3, i + 0.3],
             color='#222222', linewidth=1.5, zorder=4)

ax2.axvline(0.5, color='#999999', linestyle='--', linewidth=1)
ax2.set_yticks(range(len(datasets_sorted)))
ax2.set_yticklabels(datasets_sorted, fontsize=9)
ax2.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax2.set_title(f'Run-level spread  (n={int(c2b_agg["n"].iloc[0])} runs per dataset)', fontsize=10)
ax2.set_xlim(0, max(0.7, c2b_df['nl'].max() + 0.05))
ax2.spines[['top', 'right']].set_visible(False)

legend_patches = [
    mpatches.Patch(color='#E05A2B', label='Oncology'),
    mpatches.Patch(color='#534AB7', label='Osteoarthritis'),
    mpatches.Patch(color='#1D9E75', label='PTSD/Trauma'),
    mpatches.Patch(color='#C0A020', label='mHealth'),
    mpatches.Patch(color='#888888', label='Neurology'),
    mpatches.Patch(color='#3399CC', label='Nutrition'),
]
ax2.legend(handles=legend_patches, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('results/c2b_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Causal chain console summary 
print('\nC2b — Causal chain decomposition')
print('  C1→C2b gap = SciBERT feature contribution (classifier held constant at SVM)')
print('  C2b→C2a gap = classifier contribution (features held constant at SciBERT)\n')

try:
    chain = c2b_agg.merge(c1_agg[['dataset','mean']], on='dataset', suffixes=('_c2b','_c1'))
    chain = chain.merge(c2a_agg[['dataset','mean']], on='dataset').rename(columns={'mean':'mean_c2a'})
    chain['feat_gap']  = chain['mean_c1']  - chain['mean_c2b']   # C1→C2b
    chain['clf_gap']   = chain['mean_c2b'] - chain['mean_c2a']   # C2b→C2a
    chain['total_gap'] = chain['mean_c1']  - chain['mean_c2a']   # C1→C2a

    print(f'  {"Dataset":<28}  {"C1→C2b (feat)":>14}  {"C2b→C2a (clf)":>14}  {"C1→C2a (total)":>15}')
    print('  ' + '─' * 75)
    for row in chain.sort_values('total_gap', ascending=False).itertuples():
        print(f'  {row.dataset:<28}  {row.feat_gap:>+14.4f}  {row.clf_gap:>+14.4f}  {row.total_gap:>+15.4f}')
    print(f'\n  Pool mean  C1→C2b: {chain["feat_gap"].mean():+.4f}  |  '
          f'C2b→C2a: {chain["clf_gap"].mean():+.4f}  |  '
          f'C1→C2a: {chain["total_gap"].mean():+.4f}')
except NameError:
    print('  (Run C1 and C2a visualisation cells first to see the full causal chain)')

---
## CONDITION 3 — Graph-only GAT

This condition tests whether citation graph structure alone — without semantic text features — can produce a useful relevance ranking.

### How the citation graph is constructed

Each dataset becomes its own graph:
- **Nodes** = papers in the dataset
- **Edges** = citation links: paper A cites paper B if B's `openalex_id` appears in A's `referenced_works` list **and B is also in the dataset**

This last condition is important: we only keep edges between papers that are both inside the dataset. References to papers not in our corpus are dropped — those papers have no features.

### Node features for the GAT

The GAT layers need a numerical starting point for each node. We use:
1. **TF-IDF+LSA (128-d):** TF-IDF bag-of-words compressed with Truncated SVD to 128 dimensions. This captures basic content without SciBERT's compute cost.
2. **Structural features (4-d):** in-degree, out-degree, PageRank, publication year (normalised)

Total node feature dimension: 132-d.

### Why GAT instead of GCN

GCN uses a fixed symmetric normalisation for neighbour aggregation. GAT learns *attention weights* that determine how much each neighbour contributes to the node's updated representation. This is important for citation networks where not all citations are equally informative — a paper citing 200 unrelated works should not weight all neighbours equally.

In [ ]:
LSA_COMPONENTS = 128  # dimension of the LSA-compressed text features
GAT_HIDDEN     = 64   # hidden dimension inside each GAT layer
GAT_HEADS      = 8    # number of attention heads in the first GAT layer
GAT_OUT_DIM    = 128  # output embedding dimension from the GAT
GAT_DROPOUT    = 0.3
GAT_EPOCHS     = 200
EARLY_STOP_PAT = 20   # stop if validation loss doesn't improve for 20 epochs


def build_citation_graph(
    df: pd.DataFrame,
    directed: bool = True,
) -> tuple[torch.Tensor, dict]:
    """
    Build a PyG-compatible edge_index tensor from the referenced_works column.

    Args:
        df:       loaded dataset DataFrame (must have openalex_id, referenced_works)
        directed: if True, edges go from citing paper → cited paper (default)
                  if False, edges are made undirected (both directions added)

    Returns:
        edge_index: LongTensor of shape (2, num_edges) in COO format
        stats:      dict with graph statistics for logging
    """
    # Build a mapping from openalex_id → integer row index in the DataFrame
    # This is how we convert string IDs to node indices for the edge tensor
    id_to_idx = {oid: idx for idx, oid in enumerate(df['openalex_id'])}

    src_nodes, dst_nodes = [], []
    edges_skipped = 0  # references that point outside the dataset

    for src_idx, row in df.iterrows():
        for ref_id in row['referenced_works']:
            if ref_id in id_to_idx:
                # Edge: the paper at src_idx cites the paper at id_to_idx[ref_id]
                src_nodes.append(id_to_idx[row['openalex_id']])
                dst_nodes.append(id_to_idx[ref_id])
            else:
                edges_skipped += 1  # reference points outside this dataset

    if len(src_nodes) == 0:
        # Dataset has no internal citations — return an empty edge_index
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(
            [src_nodes, dst_nodes], dtype=torch.long
        )

        if not directed:
            # to_undirected adds reverse edges: if A→B exists, also adds B→A
            # and removes duplicates
            edge_index = to_undirected(edge_index)

        # Add self-loops: each node is its own neighbour. This prevents
        # isolated nodes from receiving zero-information after aggregation
        # (they still have their own features).
        n_real_edges = edge_index.shape[1]
        edge_index, _ = add_self_loops(edge_index, num_nodes=len(df))

    stats = {
        'n_nodes':        len(df),
        'n_edges':        n_real_edges if len(src_nodes) > 0 else 0,
        'edges_skipped':  edges_skipped,
        'density':        (n_real_edges if len(src_nodes) > 0 else 0) / (len(df) ** 2) if len(df) > 0 else 0,
    }

    return edge_index, stats


def build_lsa_features(
    train_dfs: list[pd.DataFrame],
    test_df:   pd.DataFrame,
    n_components: int = LSA_COMPONENTS,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Fit TF-IDF + TruncatedSVD (LSA) on training texts, transform test.

    TruncatedSVD applied to a TF-IDF matrix is called Latent Semantic Analysis (LSA).
    It compresses the sparse high-dimensional TF-IDF vectors into dense low-dimensional
    vectors that capture latent topics. This is efficient as node features for the GNN.

    Returns:
        train_feats: (N_train, n_components)
        test_feats:  (N_test,  n_components)
    """
    all_train_texts = [t for df in train_dfs for t in get_texts(df)]
    test_texts      = get_texts(test_df)

    # Note: stop_words='english' is kept here for the LSA step.
    # This differs from C1's TF-IDF (which uses stop_words=None to
    # replicate ELAS-Ultra exactly). For LSA, removing stop words is
    # correct: common words like 'the', 'is', 'of' would otherwise
    # dominate the first singular vectors, making the topic dimensions
    # uninformative. The Week 3 PDF does not specify stop_words for
    # the LSA step, but keeping them here is the technically sound choice.
    tfidf = TfidfVectorizer(
        sublinear_tf=True,
        max_features=50_000,
        min_df=2,
        stop_words='english',
    )
    svd = TruncatedSVD(n_components=n_components, random_state=SEED)

    # Fit on training texts only — no test data leakage
    X_train_tfidf = tfidf.fit_transform(all_train_texts)  # sparse
    X_train_lsa   = svd.fit_transform(X_train_tfidf)      # dense (N_train, 128)

    # Transform test using the *same* fitted vectoriser and SVD
    X_test_tfidf  = tfidf.transform(test_texts)
    X_test_lsa    = svd.transform(X_test_tfidf)

    return X_train_lsa, X_test_lsa


def compute_structural_features(df: pd.DataFrame, edge_index: torch.Tensor) -> np.ndarray:
    """
    Compute structural node features from the citation graph:
    in-degree, out-degree, PageRank, publication year (normalised).

    These features tell the GAT about a paper's role in the citation network
    independently of its text content. A highly cited paper (high in-degree)
    is more likely to be a foundational relevant work.

    Returns:
        np.ndarray of shape (N, 4)
    """
    N = len(df)
    feats = np.zeros((N, 4), dtype=np.float32)

    if edge_index.shape[1] > 0:
        # Build networkx graph for PageRank computation
        edges = edge_index.T.numpy()
        G = nx.DiGraph()
        G.add_nodes_from(range(N))
        G.add_edges_from(edges)

        # In-degree = how many papers cite this paper
        in_deg  = np.array([G.in_degree(i) for i in range(N)], dtype=np.float32)
        # Out-degree = how many papers this paper cites (within dataset)
        out_deg = np.array([G.out_degree(i) for i in range(N)], dtype=np.float32)
        # PageRank = eigenvector centrality, weights by importance of citing papers
        pr_dict = nx.pagerank(G, alpha=0.85)
        pagerank = np.array([pr_dict.get(i, 0.0) for i in range(N)], dtype=np.float32)

        feats[:, 0] = in_deg
        feats[:, 1] = out_deg
        feats[:, 2] = pagerank

    # Publication year: normalise to [0, 1] over the dataset's range
    years = df['publication_year'].fillna(df['publication_year'].median()).values.astype(float)
    yr_min, yr_max = years.min(), years.max()
    if yr_max > yr_min:
        feats[:, 3] = (years - yr_min) / (yr_max - yr_min)

    # L2-normalise each feature column independently to put them on the same scale
    # before concatenating with LSA features
    for j in range(feats.shape[1]):
        col_max = feats[:, j].max()
        if col_max > 0:
            feats[:, j] /= col_max

    return feats


print('Citation graph and feature extraction utilities defined.')

In [ ]:
class GATEncoder(nn.Module):
    """
    Two-layer Graph Attention Network producing node embeddings.

    Architecture:
        Input features (132-d)
        → GAT Layer 1 (8 heads × 64-d = 512-d, ELU activation, dropout)
        → GAT Layer 2 (1 head × 128-d, no activation — raw embedding)

    The 'heads' in the first layer run in parallel: each head learns a
    different attention pattern, their outputs are concatenated, giving
    8*64 = 512-d after the first layer. The second layer collapses back
    to a single 128-d vector per node.

    This output is the 'GAT node embedding' that gets concatenated with
    SciBERT embeddings in Condition 4.
    """
    def __init__(self, in_channels: int, hidden: int = GAT_HIDDEN,
                 heads: int = GAT_HEADS, out_dim: int = GAT_OUT_DIM,
                 dropout: float = GAT_DROPOUT):
        super().__init__()
        self.dropout = dropout

        # Layer 1: multi-head attention, outputs heads * hidden features per node
        self.conv1 = GATConv(
            in_channels, hidden,
            heads=heads,
            dropout=dropout,
            concat=True,   # concatenate head outputs → out_dim = heads * hidden
        )

        # Layer 2: single head, outputs out_dim features per node
        self.conv2 = GATConv(
            hidden * heads, out_dim,
            heads=1,
            dropout=dropout,
            concat=False,  # no concatenation needed for single head
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        x:          (N, in_channels) node feature matrix
        edge_index: (2, E) edge list in COO format

        Returns: (N, out_dim) node embeddings
        """
        # Dropout on input features — regularises during training
        x = F.dropout(x, p=self.dropout, training=self.training)

        # First GAT layer: each node aggregates from its neighbours
        # ELU is preferred over ReLU for attention-based networks
        x = F.elu(self.conv1(x, edge_index))

        x = F.dropout(x, p=self.dropout, training=self.training)

        # Second GAT layer: final node embedding
        x = self.conv2(x, edge_index)

        return x  # (N, GAT_OUT_DIM)


class GATClassifier(nn.Module):
    """
    Full Condition 3 model: GAT encoder + binary classification head.

    Takes a PyG Data object and outputs a relevance probability per node.
    """
    def __init__(self, in_channels: int):
        super().__init__()
        self.encoder = GATEncoder(in_channels)
        # Linear head: projects from 128-d GAT embedding to a single logit
        self.head = nn.Linear(GAT_OUT_DIM, 1)

    def forward(self, data: Data) -> torch.Tensor:
        z = self.encoder(data.x, data.edge_index)  # (N, 128)
        logits = self.head(z).squeeze(-1)           # (N,)
        return torch.sigmoid(logits)               # (N,) in [0, 1]


def make_pyg_data(
    df: pd.DataFrame,
    node_features: np.ndarray,
    edge_index: torch.Tensor,
    train_mask: np.ndarray = None,
    val_mask:   np.ndarray = None,
) -> Data:
    """
    Package a dataset into a PyG Data object.

    PyG's Data object holds all graph information in one place:
    - x:          node feature matrix
    - edge_index: edge list
    - y:          node labels
    - train_mask: boolean mask for training nodes
    - val_mask:   boolean mask for validation nodes
    """
    x      = torch.tensor(node_features, dtype=torch.float)
    y      = torch.tensor(get_labels(df), dtype=torch.float)

    data   = Data(x=x, edge_index=edge_index, y=y)

    if train_mask is not None:
        data.train_mask = torch.tensor(train_mask, dtype=torch.bool)
    if val_mask is not None:
        data.val_mask   = torch.tensor(val_mask,   dtype=torch.bool)

    return data


def focal_loss(
    preds: torch.Tensor,
    targets: torch.Tensor,
    alpha: float = 0.75,
    gamma: float = 2.0,
) -> torch.Tensor:
    """
    Focal Loss for handling severe class imbalance in citation screening.

    Standard binary cross-entropy treats all examples equally. When 95% of
    papers are irrelevant, the model learns to predict 'irrelevant' for
    everything and achieves 95% accuracy while being useless for screening.

    Focal loss down-weights easy examples (papers the model already classifies
    correctly with high confidence) and focuses training on hard examples
    (relevant papers that look irrelevant, or vice versa).

    alpha: weight for the positive (relevant) class. > 0.5 upweights relevant papers.
           Week 3 spec: alpha=0.75 (positive class gets 3× more weight).
    gamma: focusing parameter. 0 = standard BCE. 2 is the standard value
           recommended by Lin et al. (2017). Week 3 spec: gamma=2.0.
    Implementation note: alpha tensors are created with torch.full_like so
    they inherit the device of 'targets'. Using torch.tensor(alpha) would
    create a CPU tensor and crash at runtime on GPU (Colab Pro T4/A100).
    """
    bce  = F.binary_cross_entropy(preds, targets, reduction='none')
    pt   = torch.where(targets == 1, preds, 1 - preds)  # probability of correct class
    # torch.full_like ensures alpha tensors are on the same device as targets.
    # torch.tensor(alpha) would always be CPU and crash when targets is on GPU.
    at   = torch.where(targets == 1,
                       torch.full_like(targets, alpha),
                       torch.full_like(targets, 1 - alpha))
    loss = at * (1 - pt) ** gamma * bce
    return loss.mean()


print('GAT model architecture defined.')
print(f'  Input features: {LSA_COMPONENTS} (LSA) + 4 (structural) = {LSA_COMPONENTS + 4}')
print(f'  GAT layer 1:    {GAT_HEADS} heads × {GAT_HIDDEN} = {GAT_HEADS * GAT_HIDDEN}-d')
print(f'  GAT layer 2:    {GAT_OUT_DIM}-d output embedding')

In [ ]:
def train_gat_model(
    model:      nn.Module,
    data:       Data,
    n_epochs:   int   = GAT_EPOCHS,
    lr:         float = 1e-3,
    patience:   int   = EARLY_STOP_PAT,
    verbose:    bool  = False,
) -> nn.Module:
    """
    Train the GAT model with early stopping on validation loss.

    The entire graph is in memory at once (transductive-style inference).
    Training uses only train_mask nodes for the loss; val_mask nodes are
    used only for early stopping — their labels are never in the loss.

    Returns the model at its best validation loss checkpoint.
    """
    data  = data.to(DEVICE)
    model = model.to(DEVICE)

    optimiser = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimiser, patience=10, factor=0.5)

    best_val_loss   = float('inf')
    best_state_dict = None
    no_improve      = 0

    for epoch in range(n_epochs):
        # Training step 
        model.train()
        optimiser.zero_grad()

        probs = model(data)                         # (N,) probabilities
        train_loss = focal_loss(
            probs[data.train_mask],
            data.y[data.train_mask],
        )
        train_loss.backward()
        optimiser.step()

        # Validation step 
        model.eval()
        with torch.no_grad():
            probs_val = model(data)
            val_loss  = focal_loss(
                probs_val[data.val_mask],
                data.y[data.val_mask],
            )

        scheduler.step(val_loss)

        # Early stopping 
        if val_loss.item() < best_val_loss:
            best_val_loss   = val_loss.item()
            # deepcopy of state_dict captures model weights at this checkpoint
            best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve      = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            if verbose:
                print(f'    Early stop at epoch {epoch+1} (best val loss: {best_val_loss:.4f})')
            break

        if verbose and (epoch + 1) % 20 == 0:
            print(f'    Epoch {epoch+1:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}')

    # Restore best checkpoint
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model


print('GAT training loop defined with early stopping.')

In [ ]:
def run_condition3_lodo(
    dataset_names: list[str],
    n_runs: int = 5,
    directed: bool = True,
) -> list[dict]:
    """
    Run Condition 3 (GAT graph-only) under LODO evaluation.

    For each LODO fold:
      1. Fit TF-IDF+LSA on all training texts
      2. Build the citation graph for the test dataset
      3. Combine LSA + structural features as node features
      4. Build a training 'meta-graph' from all training datasets (separate graphs
         concatenated — they don't share edges)
      5. Train the GAT on the training graphs
      6. Infer on the held-out test graph

    Note on LODO + GNN: the GAT is trained on multiple separate graphs
    (one per training dataset). At test time it runs on a completely new graph
    (the held-out dataset). This is *inductive* graph learning — the model
    must generalise its learned attention patterns to an unseen graph structure,
    not just unseen nodes on a known graph.
    """
    folds   = lodo_splits(dataset_names)
    results = []
    IN_CHANNELS = LSA_COMPONENTS + 4  # 128 LSA + 4 structural

    for fold in folds:
        test_name   = fold['test']
        train_names = fold['train']

        test_df  = DATASETS[test_name]
        train_dfs = [DATASETS[n] for n in train_names]

        # Fit LSA on training texts, transform test
        train_lsa_parts, test_lsa = build_lsa_features(train_dfs, test_df)

        # Build test graph and features
        test_edge_index, test_stats = build_citation_graph(test_df, directed=directed)
        test_struct = compute_structural_features(test_df, test_edge_index)
        test_feats  = np.hstack([test_lsa, test_struct])  # (N_test, 132)

        best_nl_for_dataset    = float('inf')
        best_state_for_dataset = None

        for run_idx in range(n_runs):
            run_seed = SEED + run_idx
            torch.manual_seed(run_seed)
            np.random.seed(run_seed)
            t0 = time.time()

            # Build one PyG Data object per training dataset, keeping them separate
            # (the graphs are independent — no cross-dataset edges)
            train_data_list = []
            lsa_offset = 0
            for i, tr_df in enumerate(train_dfs):
                N_i = len(tr_df)
                tr_lsa_i = train_lsa_parts[lsa_offset : lsa_offset + N_i]
                lsa_offset += N_i

                tr_edge_index, _ = build_citation_graph(tr_df, directed=directed)
                tr_struct        = compute_structural_features(tr_df, tr_edge_index)
                tr_feats         = np.hstack([tr_lsa_i, tr_struct])

                # Stratified 80/20 train/val split — Week 3 spec.
                # Stratification ensures both splits contain relevant papers
                # even when inclusion rates are as low as 0.2%. A plain
                # random permutation would frequently put 0 relevant papers
                # in the val set, making early stopping on val loss useless.
                n_nodes   = len(tr_df)
                tr_labels = get_labels(tr_df)
                all_idx   = np.arange(n_nodes)
                # fall back to random if only one class present in this dataset
                try:
                    train_idx, val_idx = train_test_split(
                        all_idx, test_size=0.2,
                        stratify=tr_labels, random_state=run_seed,
                    )
                except ValueError:
                    # only one class — stratify is undefined, use random split
                    train_idx, val_idx = train_test_split(
                        all_idx, test_size=0.2, random_state=run_seed,
                    )

                train_mask = np.zeros(n_nodes, dtype=bool)
                val_mask   = np.zeros(n_nodes, dtype=bool)
                train_mask[train_idx] = True
                val_mask[val_idx]     = True

                pyg_data = make_pyg_data(tr_df, tr_feats, tr_edge_index,
                                         train_mask=train_mask, val_mask=val_mask)
                train_data_list.append(pyg_data)

            # Train one model across all training graphs by iterating over them
            model = GATClassifier(in_channels=IN_CHANNELS)
            model.to(DEVICE)
            optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

            best_val = float('inf')
            no_imp   = 0
            best_state = None

            for epoch in range(GAT_EPOCHS):
                epoch_train_loss = 0.0
                epoch_val_loss   = 0.0

                for g_data in train_data_list:
                    g_data = g_data.to(DEVICE)
                    model.train()
                    optimizer.zero_grad()
                    probs = model(g_data)
                    loss  = focal_loss(probs[g_data.train_mask], g_data.y[g_data.train_mask])
                    loss.backward()
                    optimizer.step()
                    epoch_train_loss += loss.item()

                    model.eval()
                    with torch.no_grad():
                        vp = model(g_data)
                        vl = focal_loss(vp[g_data.val_mask], g_data.y[g_data.val_mask])
                    epoch_val_loss += vl.item()

                avg_val = epoch_val_loss / len(train_data_list)
                if avg_val < best_val:
                    best_val   = avg_val
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                    no_imp     = 0
                else:
                    no_imp += 1

                if no_imp >= EARLY_STOP_PAT:
                    break

            if best_state:
                model.load_state_dict(best_state)
            # Infer on the held-out test graph
            test_pyg = make_pyg_data(test_df, test_feats, test_edge_index).to(DEVICE)
            model.eval()
            with torch.no_grad():
                scores = model(test_pyg).cpu().numpy()

            test_labels = get_labels(test_df)
            nl    = normalised_loss(test_labels, scores)

            if nl < best_nl_for_dataset:
                best_nl_for_dataset    = nl
                best_state_for_dataset = best_state
            wss95 = wss_at_95(test_labels, scores)
            rt    = time.time() - t0

            log_result(
                dataset=test_name, held_out=test_name,
                model='C3_GAT_only',
                run=run_idx, seed=run_seed,
                nl=nl, wss95=wss95, runtime_s=rt,
                notes=f'directed={directed} edges={test_stats["n_edges"]}',
            )
            results.append({'dataset': test_name, 'run': run_idx, 'nl': nl})
            print(f'  C3 | fold: {test_name:<28} run {run_idx} | NL={nl:.4f} ({rt:.1f}s)')

        # Save the best model state across all runs for this dataset
        if best_state_for_dataset:
            cache_path = GAT_CACHE_DIR / f'c3_{test_name}.pt'
            torch.save(best_state_for_dataset, cache_path)
            print(f'  C3 | saved best model for {test_name} (NL={best_nl_for_dataset:.4f})')

    return results


print('Condition 3 (GAT-only) ready.')

In [ ]:
print(f'Running Condition 3 (GAT-only) on {len(rq1_names)} datasets × {N_RUNS} runs...')
c3_results = run_condition3_lodo(rq1_names, n_runs=N_RUNS, directed=True)

c3_df = pd.DataFrame(c3_results)
print('\nCondition 3 — Mean NL per dataset:')
print(c3_df.groupby('dataset')['nl'].mean().sort_values().to_string())

In [ ]:
# C3 Results Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

c3_agg = (
    c3_df.groupby('dataset')['nl']
    .agg(mean='mean', std='std', n='count')
    .reset_index()
    .sort_values('mean')
)

DOMAIN_COLOURS = {
    'Greca_2023':           '#E05A2B',
    'Erer_2015':            '#E05A2B',
    'Eggmann_2023':         '#E05A2B',
    'Pijls_2018':           '#534AB7',
    'Pinos-Cisneros_2023':  '#534AB7',
    'Tumkaya_2018':         '#1D9E75',
    'Rinne_2021':           '#1D9E75',
    'Abgaz_2023':           '#C0A020',
    'Demirkaya_2015':       '#888888',
    'Leenaars_2020':        '#3399CC',
}

colours = [DOMAIN_COLOURS.get(d, '#AAAAAA') for d in c3_agg['dataset']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Condition 3 — GAT-only  [citation graph structure, no text] (LODO)',
             fontsize=13, fontweight='bold', y=1.01)

# Left: mean NL per dataset 
ax = axes[0]
x = np.arange(len(c3_agg))

ax.barh(x, c3_agg['mean'], xerr=c3_agg['std'],
        color=colours, alpha=0.85, capsize=4,
        error_kw={'linewidth': 1.0, 'ecolor': '#333333'})

# C1 overlay — the key comparison: does graph structure alone beat text?
try:
    for i, ds in enumerate(c3_agg['dataset']):
        val = c1_agg.loc[c1_agg['dataset'] == ds, 'mean'].values
        if len(val):
            ax.scatter(val[0], i, marker='D', s=40, color='#CC2222',
                       zorder=5, label='C1 (TF-IDF+SVM)' if i == 0 else '')
    ax.legend(fontsize=8, loc='lower right')
except NameError:
    pass

ax.axvline(0.5, color='#999999', linestyle='--', linewidth=1)
ax.axvline(c3_agg['mean'].mean(), color='#D85A30', linestyle='-',
           linewidth=1.5, label=f'C3 mean ({c3_agg["mean"].mean():.3f})')

ax.set_yticks(x)
ax.set_yticklabels(c3_agg['dataset'], fontsize=9)
ax.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax.set_title('Mean NL per dataset  ±1 SD across runs\n◆ = C1 baseline', fontsize=9)
ax.set_xlim(0, max(0.7, c3_agg['mean'].max() + c3_agg['std'].max() + 0.05))
ax.spines[['top', 'right']].set_visible(False)

for i, row in enumerate(c3_agg.itertuples()):
    ax.text(row.mean + row.std + 0.01, i, f'{row.mean:.3f}',
            va='center', fontsize=8, color='#333333')

# Right: run variance — C3 should show real spread unlike C1 
ax2 = axes[1]

datasets_sorted = c3_agg['dataset'].tolist()
for i, ds in enumerate(datasets_sorted):
    vals = c3_df[c3_df['dataset'] == ds]['nl'].values
    jitter = np.random.uniform(-0.15, 0.15, size=len(vals))
    ax2.scatter(vals, np.full_like(vals, i) + jitter,
                color=DOMAIN_COLOURS.get(ds, '#AAAAAA'),
                alpha=0.7, s=40, zorder=3)
    ax2.plot([np.median(vals), np.median(vals)], [i - 0.3, i + 0.3],
             color='#222222', linewidth=1.5, zorder=4)

ax2.axvline(0.5, color='#999999', linestyle='--', linewidth=1)
ax2.set_yticks(range(len(datasets_sorted)))
ax2.set_yticklabels(datasets_sorted, fontsize=9)
ax2.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax2.set_title(f'Run-level spread  (n={int(c3_agg["n"].iloc[0])} runs per dataset)\n'
              'Wider spread = GAT more sensitive to weight init / train split', fontsize=9)
ax2.set_xlim(0, max(0.7, c3_df['nl'].max() + 0.05))
ax2.spines[['top', 'right']].set_visible(False)

legend_patches = [
    mpatches.Patch(color='#E05A2B', label='Oncology'),
    mpatches.Patch(color='#534AB7', label='Osteoarthritis'),
    mpatches.Patch(color='#1D9E75', label='PTSD/Trauma'),
    mpatches.Patch(color='#C0A020', label='mHealth'),
    mpatches.Patch(color='#888888', label='Neurology'),
    mpatches.Patch(color='#3399CC', label='Nutrition'),
]
ax2.legend(handles=legend_patches, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('results/c3_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Console summary 
print('\nC3 — Normalised Loss summary')
print(f'  Pool mean ± std : {c3_agg["mean"].mean():.4f} ± {c3_agg["mean"].std():.4f}')
print(f'  Best dataset    : {c3_agg.iloc[0]["dataset"]}  ({c3_agg.iloc[0]["mean"]:.4f})')
print(f'  Worst dataset   : {c3_agg.iloc[-1]["dataset"]}  ({c3_agg.iloc[-1]["mean"]:.4f})')
print(f'\n  Run variance (std across 5 runs) — non-zero expected for GAT:')
for row in c3_agg.sort_values('std', ascending=False).itertuples():
    print(f'  {row.dataset:<28}  mean={row.mean:.4f}  std={row.std:.4f}')

try:
    merged = c3_agg.merge(c1_agg[['dataset', 'mean']], on='dataset', suffixes=('_c3', '_c1'))
    merged['delta'] = merged['mean_c1'] - merged['mean_c3']
    print(f'\n  C1 → C3 delta  (positive = graph structure helps over TF-IDF):')
    for row in merged.sort_values('delta', ascending=False).itertuples():
        arrow = '▲' if row.delta > 0 else '▼'
        print(f'  {row.dataset:<28}  {arrow} {abs(row.delta):.4f}')
    print(f'\n  Mean delta: {merged["delta"].mean():+.4f}  '
          f'({"graph helps" if merged["delta"].mean() > 0 else "TF-IDF still better"} on average)')
except NameError:
    print('\n  (Run C1 visualisation cell first to see C1 vs C3 comparison)')

---
## CONDITION 4 — GNN + NLP Fusion (main model)

This is the thesis's proposed model. It concatenates the SciBERT [CLS] embedding (768-d) with the GAT node embedding (128-d) to form a 896-d joint representation for each paper.

The fusion is early: both branches run in parallel, their outputs are concatenated, and a shared classifier head is trained on the combined vector. This lets the classifier learn to weight text features and graph features differently depending on which is more informative for a given paper.

Architecture recap:
```
SciBERT (frozen) → [CLS] 768-d  ──┐
                                   ├── concat → 896-d → Linear → sigmoid → p(relevant)
GAT (trainable)  → embed 128-d  ──┘
```

In [ ]:
SCIBERT_DIM = 768
FUSION_DIM  = SCIBERT_DIM + GAT_OUT_DIM  # 768 + 128 = 896


class FusionClassifier(nn.Module):
    """
    GNN + NLP Fusion model (Condition 4).

    The GAT encoder operates on node features derived from the citation graph.
    Its output is concatenated with the pre-computed frozen SciBERT embeddings.
    A two-layer MLP head then maps the 896-d joint representation to a
    relevance probability.

    Note: SciBERT embeddings are passed in as fixed tensors, not computed here.
    The model only holds the GAT encoder and the classifier head.
    """
    def __init__(self, gat_in_channels: int):
        super().__init__()
        # GAT encoder — same architecture as Condition 3
        self.gat = GATEncoder(in_channels=gat_in_channels)

        # Fusion head: two-layer MLP with dropout
        # 896 → 256 → 1
        self.head = nn.Sequential(
            nn.Linear(FUSION_DIM, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, 1),
        )

    def forward(
        self,
        data: Data,
        scibert_embs: torch.Tensor,  # (N, 768) pre-computed, frozen
    ) -> torch.Tensor:
        """
        data:          PyG Data with x (LSA+structural features) and edge_index
        scibert_embs:  SciBERT [CLS] embeddings for the same nodes (frozen)

        Returns: (N,) relevance probabilities
        """
        # GAT forward pass on the citation graph structure
        gat_embs = self.gat(data.x, data.edge_index)  # (N, 128)

        # Concatenate along the feature dimension
        # dim=1: [scibert_embs (768) | gat_embs (128)] → (N, 896)
        joint = torch.cat([scibert_embs, gat_embs], dim=1)

        # Classify
        logits = self.head(joint).squeeze(-1)  # (N,)
        return torch.sigmoid(logits)


def run_condition4_lodo(
    dataset_names: list[str],
    n_runs: int = 5,
    directed: bool = True,
) -> list[dict]:
    """
    Run Condition 4 (GNN+NLP Fusion) under LODO evaluation.

    Identical structure to Condition 3, but the model receives both
    the citation graph and the SciBERT embeddings.
    """
    folds      = lodo_splits(dataset_names)
    results    = []
    GAT_IN_DIM = LSA_COMPONENTS + 4  # same as Condition 3

    for fold in folds:
        test_name   = fold['test']
        train_names = fold['train']

        test_df   = DATASETS[test_name]
        train_dfs = [DATASETS[n] for n in train_names]

        # LSA features (same as C3)
        train_lsa_parts, test_lsa = build_lsa_features(train_dfs, test_df)

        # Test graph
        test_edge_index, test_stats = build_citation_graph(test_df, directed=directed)
        test_struct = compute_structural_features(test_df, test_edge_index)
        test_feats  = np.hstack([test_lsa, test_struct])

        # SciBERT embeddings for test dataset (pre-computed, frozen)
        test_scibert = torch.tensor(
            SCIBERT_EMBEDDINGS[test_name], dtype=torch.float
        ).to(DEVICE)

        for run_idx in range(n_runs):
            run_seed = SEED + run_idx
            torch.manual_seed(run_seed)
            np.random.seed(run_seed)
            t0 = time.time()

            # Build training graphs (same structure as C3)
            train_data_list   = []
            train_scibert_list = []
            lsa_offset = 0

            for i, tr_df in enumerate(train_dfs):
                N_i = len(tr_df)
                tr_lsa_i = train_lsa_parts[lsa_offset : lsa_offset + N_i]
                lsa_offset += N_i

                tr_edge_index, _ = build_citation_graph(tr_df, directed=directed)
                tr_struct        = compute_structural_features(tr_df, tr_edge_index)
                tr_feats         = np.hstack([tr_lsa_i, tr_struct])

                # Stratified 80/20 train/val split — Week 3 spec.
                # Same rationale as C3: stratification guarantees relevant
                # papers appear in the val set for reliable early stopping.
                n_nodes   = len(tr_df)
                tr_labels = get_labels(tr_df)
                all_idx   = np.arange(n_nodes)
                try:
                    train_idx, val_idx = train_test_split(
                        all_idx, test_size=0.2,
                        stratify=tr_labels, random_state=run_seed,
                    )
                except ValueError:
                    train_idx, val_idx = train_test_split(
                        all_idx, test_size=0.2, random_state=run_seed,
                    )
                train_mask = np.zeros(n_nodes, dtype=bool)
                val_mask   = np.zeros(n_nodes, dtype=bool)
                train_mask[train_idx] = True
                val_mask[val_idx]     = True

                pyg_data = make_pyg_data(tr_df, tr_feats, tr_edge_index,
                                         train_mask=train_mask, val_mask=val_mask)
                train_data_list.append(pyg_data)

                # SciBERT embeddings for this training dataset
                tr_scibert = torch.tensor(
                    SCIBERT_EMBEDDINGS[train_names[i]], dtype=torch.float
                )
                train_scibert_list.append(tr_scibert)

            # Train fusion model
            model     = FusionClassifier(gat_in_channels=GAT_IN_DIM).to(DEVICE)
            optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

            best_val = float('inf')
            no_imp   = 0
            best_state = None

            for epoch in range(GAT_EPOCHS):
                epoch_val = 0.0

                for g_data, g_scibert in zip(train_data_list, train_scibert_list):
                    g_data    = g_data.to(DEVICE)
                    g_scibert = g_scibert.to(DEVICE)

                    model.train()
                    optimizer.zero_grad()
                    probs = model(g_data, g_scibert)
                    loss  = focal_loss(probs[g_data.train_mask], g_data.y[g_data.train_mask])
                    loss.backward()
                    optimizer.step()

                    model.eval()
                    with torch.no_grad():
                        vp = model(g_data, g_scibert)
                        vl = focal_loss(vp[g_data.val_mask], g_data.y[g_data.val_mask])
                    epoch_val += vl.item()

                avg_val = epoch_val / len(train_data_list)
                if avg_val < best_val:
                    best_val   = avg_val
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                    no_imp     = 0
                else:
                    no_imp += 1

                if no_imp >= EARLY_STOP_PAT:
                    break

            if best_state:
                model.load_state_dict(best_state)

            # Infer on test graph
            test_pyg = make_pyg_data(test_df, test_feats, test_edge_index).to(DEVICE)
            model.eval()
            with torch.no_grad():
                scores = model(test_pyg, test_scibert).cpu().numpy()

            test_labels = get_labels(test_df)
            nl    = normalised_loss(test_labels, scores)
            wss95 = wss_at_95(test_labels, scores)
            rt    = time.time() - t0

            log_result(
                dataset=test_name, held_out=test_name,
                model='C4_GNN_NLP_Fusion',
                run=run_idx, seed=run_seed,
                nl=nl, wss95=wss95, runtime_s=rt,
                notes=f'directed={directed} fusion_dim={FUSION_DIM}',
            )
            results.append({'dataset': test_name, 'run': run_idx, 'nl': nl})
            print(f'  C4 | fold: {test_name:<28} run {run_idx} | NL={nl:.4f} ({rt:.1f}s)')

    return results


print(f'Condition 4 (GNN+NLP Fusion) ready. Fusion dim = {FUSION_DIM}')

In [ ]:
print(f'Running Condition 4 (GNN+NLP Fusion) on {len(rq1_names)} datasets × {N_RUNS} runs...')
c4_results = run_condition4_lodo(rq1_names, n_runs=N_RUNS, directed=True)

c4_df = pd.DataFrame(c4_results)
print('\nCondition 4 — Mean NL per dataset:')
print(c4_df.groupby('dataset')['nl'].mean().sort_values().to_string())

In [ ]:
# C4 Results Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

c4_agg = (
    c4_df.groupby('dataset')['nl']
    .agg(mean='mean', std='std', n='count')
    .reset_index()
    .sort_values('mean')
)

DOMAIN_COLOURS = {
    'Greca_2023':           '#E05A2B',
    'Erer_2015':            '#E05A2B',
    'Eggmann_2023':         '#E05A2B',
    'Pijls_2018':           '#534AB7',
    'Pinos-Cisneros_2023':  '#534AB7',
    'Tumkaya_2018':         '#1D9E75',
    'Rinne_2021':           '#1D9E75',
    'Abgaz_2023':           '#C0A020',
    'Demirkaya_2015':       '#888888',
    'Leenaars_2020':        '#3399CC',
}

colours = [DOMAIN_COLOURS.get(d, '#AAAAAA') for d in c4_agg['dataset']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Condition 4 — GNN+NLP Fusion  [citation graph + SciBERT text] (LODO)',
             fontsize=13, fontweight='bold', y=1.01)

# Left: mean NL per dataset
ax = axes[0]
x = np.arange(len(c4_agg))
ax.barh(x, c4_agg['mean'], xerr=c4_agg['std'],
        color=colours, alpha=0.85, capsize=4,
        error_kw={'linewidth': 1.0, 'ecolor': '#333333'})

# C2a overlay — key comparison: does fusion beat the best text-only baseline?
try:
    for i, ds in enumerate(c4_agg['dataset']):
        val = c2a_agg.loc[c2a_agg['dataset'] == ds, 'mean'].values
        if len(val):
            ax.scatter(val[0], i, marker='D', s=40, color='#3182bd',
                       zorder=5, label='C2a (SciBERT+LR)' if i == 0 else '')
    ax.legend(fontsize=8, loc='lower right')
except NameError:
    pass

# C1 overlay — secondary comparison against TF-IDF baseline
try:
    for i, ds in enumerate(c4_agg['dataset']):
        val = c1_agg.loc[c1_agg['dataset'] == ds, 'mean'].values
        if len(val):
            ax.scatter(val[0], i, marker='v', s=40, color='#CC2222',
                       zorder=5, label='C1 (TF-IDF+SVM)' if i == 0 else '')
except NameError:
    pass

ax.legend(fontsize=8, loc='lower right')

ax.axvline(0.5, color='#999999', linestyle='--', linewidth=1)
ax.axvline(c4_agg['mean'].mean(), color='#e6550d', linestyle='-',
           linewidth=1.5, label=f'C4 mean ({c4_agg["mean"].mean():.3f})')
ax.set_yticks(x)
ax.set_yticklabels(c4_agg['dataset'], fontsize=9)
ax.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax.set_title('Mean NL per dataset  ±1 SD across runs\n◆ = C2a  ▼ = C1  baselines', fontsize=9)
ax.set_xlim(0, max(0.7, c4_agg['mean'].max() + c4_agg['std'].max() + 0.05))
ax.spines[['top', 'right']].set_visible(False)

for i, row in enumerate(c4_agg.itertuples()):
    ax.text(row.mean + row.std + 0.01, i, f'{row.mean:.3f}',
            va='center', fontsize=8, color='#333333')

# Right: run variance
ax2 = axes[1]
datasets_sorted = c4_agg['dataset'].tolist()

for i, ds in enumerate(datasets_sorted):
    vals = c4_df[c4_df['dataset'] == ds]['nl'].values
    jitter = np.random.uniform(-0.15, 0.15, size=len(vals))
    ax2.scatter(vals, np.full_like(vals, i) + jitter,
                color=DOMAIN_COLOURS.get(ds, '#AAAAAA'),
                alpha=0.7, s=40, zorder=3)
    ax2.plot([np.median(vals), np.median(vals)], [i - 0.3, i + 0.3],
             color='#222222', linewidth=1.5, zorder=4)

ax2.axvline(0.5, color='#999999', linestyle='--', linewidth=1)
ax2.set_yticks(range(len(datasets_sorted)))
ax2.set_yticklabels(datasets_sorted, fontsize=9)
ax2.set_xlabel('Normalised Loss  (lower = better)', fontsize=10)
ax2.set_title(f'Run-level spread  (n={int(c4_agg["n"].iloc[0])} runs per dataset)\n'
              'Wider spread = fusion more sensitive to weight init / train split', fontsize=9)
ax2.set_xlim(0, max(0.7, c4_df['nl'].max() + 0.05))
ax2.spines[['top', 'right']].set_visible(False)

legend_patches = [
    mpatches.Patch(color='#E05A2B', label='Oncology'),
    mpatches.Patch(color='#534AB7', label='Osteoarthritis'),
    mpatches.Patch(color='#1D9E75', label='PTSD/Trauma'),
    mpatches.Patch(color='#C0A020', label='mHealth'),
    mpatches.Patch(color='#888888', label='Neurology'),
    mpatches.Patch(color='#3399CC', label='Nutrition'),
]
ax2.legend(handles=legend_patches, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('results/c4_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Console summary
print('\nC4 — Normalised Loss summary')
print(f'  Pool mean ± std : {c4_agg["mean"].mean():.4f} ± {c4_agg["mean"].std():.4f}')
print(f'  Best dataset    : {c4_agg.iloc[0]["dataset"]}  ({c4_agg.iloc[0]["mean"]:.4f})')
print(f'  Worst dataset   : {c4_agg.iloc[-1]["dataset"]}  ({c4_agg.iloc[-1]["mean"]:.4f})')
print(f'\n  Run variance (std across runs) — non-zero expected for GNN:')
for row in c4_agg.sort_values('std', ascending=False).itertuples():
    print(f'  {row.dataset:<28}  mean={row.mean:.4f}  std={row.std:.4f}')

try:
    merged = c4_agg.merge(c2a_agg[['dataset', 'mean']], on='dataset', suffixes=('_c4', '_c2a'))
    merged['delta'] = merged['mean_c2a'] - merged['mean_c4']
    print(f'\n  C2a → C4 delta  (positive = fusion beats best text-only baseline):')
    for row in merged.sort_values('delta', ascending=False).itertuples():
        arrow = '▲' if row.delta > 0 else '▼'
        print(f'  {row.dataset:<28}  {arrow} {abs(row.delta):.4f}')
    print(f'\n  Mean delta: {merged["delta"].mean():+.4f}  '
          f'({"fusion helps" if merged["delta"].mean() > 0 else "C2a still better"} on average)')
except NameError:
    print('\n  (Run C2a visualisation cell first to see C2a vs C4 comparison)')

---
## Results & Visualisation

### RQ1: Comparing all four conditions

In [ ]:
def plot_rq1_comparison(results_path: Path = RESULTS_PATH) -> None:
    """
    Plot mean Normalised Loss per dataset × condition.

    The plot shows a grouped bar chart — one group per dataset, four bars
    per group (one per condition). Error bars show ±1 standard deviation
    across the N_RUNS repetitions.
    """
    df = pd.read_csv(results_path)

    # Filter to RQ1 datasets and the four conditions only
    rq1_mask = df['dataset'].isin(rq1_names)
    cond_mask = df['model'].isin(['C1_TFIDF_SVM', 'C2b_SciBERT_SVM',
                               'C2a_SciBERT_LR', 'C3_GAT_only',
                               'C4_GNN_NLP_Fusion'])
    df = df[rq1_mask & cond_mask]

    pivot_mean = df.groupby(['dataset', 'model'])['nl'].mean().unstack()
    pivot_std  = df.groupby(['dataset', 'model'])['nl'].std().unstack()

    # Rename columns for the legend
    col_names = {
    'C1_TFIDF_SVM':      'C1: TF-IDF+SVM',
    'C2b_SciBERT_SVM':   'C2b: SciBERT+SVM',
    'C2a_SciBERT_LR':   'C2a: SciBERT+LR',
    'C3_GAT_only':       'C3: GAT-only',
    'C4_GNN_NLP_Fusion': 'C4: GNN+NLP Fusion',
    }   
    
    pivot_mean = pivot_mean.rename(columns=col_names)
    pivot_std  = pivot_std.rename(columns=col_names)

    n_datasets = len(pivot_mean)
    n_conds    = len(pivot_mean.columns)
    bar_width  = 0.18
    x          = np.arange(n_datasets)

    colours = ['#534AB7', '#9FE1CB', '#1D9E75', '#D85A30', '#EF9F27']

    fig, ax = plt.subplots(figsize=(14, 5))

    for i, (col, colour) in enumerate(zip(pivot_mean.columns, colours)):
        offset = (i - n_conds / 2 + 0.5) * bar_width
        ax.bar(
            x + offset,
            pivot_mean[col],
            width=bar_width,
            label=col,
            color=colour,
            alpha=0.85,
            yerr=pivot_std[col],
            capsize=3,
            error_kw={'linewidth': 0.8},
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [d.replace('_', '\n') for d in pivot_mean.index],
        fontsize=8,
    )
    ax.set_ylabel('Normalised Loss (lower = better)', fontsize=10)
    ax.set_title('RQ1: Four-condition ablation — LODO Normalised Loss per dataset', fontsize=11)
    ax.legend(fontsize=9, loc='upper right')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)

    plt.tight_layout()
    plt.savefig('results/rq1_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/rq1_comparison.png')


plot_rq1_comparison()

In [ ]:
def print_rq1_summary_table(results_path: Path = RESULTS_PATH) -> None:
    """
    Print the main results table for the thesis (Table 2).

    Rows = datasets, Columns = conditions, Cells = mean NL ± std.
    The bottom row shows the overall mean across all datasets.
    """
    df = pd.read_csv(results_path)
    df = df[df['dataset'].isin(rq1_names)]

    # C2b is included so the table shows the full causal chain:
    # C1→C2b gap = SciBERT features; C2b→C2a gap = classifier choice
    cond_order = ['C1_TFIDF_SVM', 'C2a_SciBERT_LR', 'C2b_SciBERT_SVM',
                  'C3_GAT_only', 'C4_GNN_NLP_Fusion']
    cond_labels = ['C1 TF-IDF+SVM', 'C2a SciBERT+LR', 'C2b SciBERT+SVM',
                   'C3 GAT', 'C4 Fusion']

    rows = []
    for ds in rq1_names:
        row = {'Dataset': ds}
        for cond, label in zip(cond_order, cond_labels):
            sub = df[(df['dataset'] == ds) & (df['model'] == cond)]['nl']
            if len(sub) > 0:
                row[label] = f'{sub.mean():.3f} ±{sub.std():.3f}'
            else:
                row[label] = '—'
        rows.append(row)

    # Overall mean row
    overall = {'Dataset': 'MEAN'}
    for cond, label in zip(cond_order, cond_labels):
        sub = df[df['model'] == cond]['nl']
        overall[label] = f'{sub.mean():.3f}' if len(sub) > 0 else '—'
    rows.append(overall)

    summary = pd.DataFrame(rows).set_index('Dataset')
    print('\nRQ1 Results — Mean Normalised Loss ± Std (lower is better)\n')
    print(summary.to_string())
    summary.to_csv('results/rq1_summary_table.csv')
    print('\nSaved: results/rq1_summary_table.csv')


print_rq1_summary_table()

In [ ]:
# RQ1 Statistical Analysis 
# Compares all five ablation conditions across 10 RQ1 datasets under LODO.
#
# Unit of analysis: dataset-level mean NL (averaged across N_RUNS seeds).
# N = 10 datasets — enough for Friedman + Wilcoxon, still small so report
# effect sizes alongside p-values.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from scipy import stats
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

RESULTS_PATH = Path('results/all_results.csv')

CONDITIONS = [
    'C1_TFIDF_SVM',
    'C2a_SciBERT_LR',
    'C2b_SciBERT_SVM',
    'C3_GAT_only',
    'C4_GNN_NLP_Fusion',
]
CONDITION_LABELS = {
    'C1_TFIDF_SVM':      'C1 TF-IDF+SVM',
    'C2a_SciBERT_LR':    'C2a SciBERT+LR',
    'C2b_SciBERT_SVM':   'C2b SciBERT+SVM',
    'C3_GAT_only':       'C3 GAT-only',
    'C4_GNN_NLP_Fusion': 'C4 GNN+NLP',
}

RQ1_DATASETS = [d[0] for d in [
    ('Greca_2023',          'oncology',       10),
    ('Pijls_2018',          'osteoarthritis', 10),
    ('Tumkaya_2018',        'ptsd_trauma',     9),
    ('Rinne_2021',          'ptsd_trauma',     9),
    ('Pinos-Cisneros_2023', 'osteoarthritis',  8),
    ('Abgaz_2023',          'mhealth',         8),
    ('Erer_2015',           'oncology',        8),
    ('Demirkaya_2015',      'neurology',       8),
    ('Leenaars_2020',       'nutrition',       8),
    ('Eggmann_2023',        'oncology',        8),
]]

# Helpers (same as RQ2 cell; safe to redefine) 

def rank_biserial(x, y):
    d = np.array(x) - np.array(y)
    d = d[d != 0]
    if len(d) == 0:
        return np.nan
    ranks = stats.rankdata(np.abs(d))
    r_plus  = np.sum(ranks[d > 0])
    r_minus = np.sum(ranks[d < 0])
    n = len(d)
    return (r_plus - r_minus) / (n * (n + 1) / 2)

def holm_bonferroni(p_values):
    n = len(p_values)
    order = np.argsort(p_values)
    corrected = np.zeros(n)
    for rank, idx in enumerate(order):
        corrected[idx] = min(p_values[idx] * (n - rank), 1.0)
    for i in range(1, n):
        corrected[order[i]] = max(corrected[order[i]], corrected[order[i - 1]])
    return corrected.tolist()

def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# Load & pivot to dataset-level means 

all_results = pd.read_csv(RESULTS_PATH)
pivot = (
    all_results
    .groupby(['dataset', 'model'])['nl']
    .mean()
    .reset_index()
    .pivot(index='dataset', columns='model', values='nl')
)

rq1 = pivot.loc[pivot.index.isin(RQ1_DATASETS), CONDITIONS].dropna()
N = len(rq1)
print(f'RQ1 datasets with complete data: {N}')
print(f'  {list(rq1.index)}')

# Descriptive statistics 

print('\n' + '═' * 70)
print('  RQ1 DESCRIPTIVE STATISTICS — Normalised Loss (↓ better)')
print('═' * 70)
print(f'  {"Condition":<22} {"Mean":>7}  {"Std":>7}  {"Min":>7}  {"Max":>7}  {"Median":>8}')
print('  ' + '-' * 64)
for cond in CONDITIONS:
    v = rq1[cond]
    print(f'  {CONDITION_LABELS[cond]:<22} {v.mean():7.4f}  {v.std():7.4f}  '
          f'{v.min():7.4f}  {v.max():7.4f}  {v.median():8.4f}')

# Friedman test 

print('\n' + '═' * 70)
print('  FRIEDMAN TEST  H₀: all five conditions are equivalent')
print('═' * 70)
groups = [rq1[c].values for c in CONDITIONS]
stat, p = friedmanchisquare(*groups)
print(f'\n  χ²({len(CONDITIONS)-1}) = {stat:.4f},  p = {p:.6f}  {sig_stars(p)}')
print(f'  N = {N} datasets  ×  {len(CONDITIONS)} conditions')
if p < 0.05:
    print('  → Reject H₀: at least one condition differs. Proceed to post-hoc.')
else:
    print('  → Fail to reject H₀ (interpret post-hoc with caution).')

# All-pairs Wilcoxon signed-rank (Holm–Bonferroni)

print('\n' + '═' * 70)
print('  ALL-PAIRS WILCOXON SIGNED-RANK TESTS (Holm–Bonferroni corrected)')
print('  Effect size r: |r|≥0.1 small, ≥0.3 medium, ≥0.5 large')
print('═' * 70)

pairs = list(combinations(CONDITIONS, 2))
raw_p, pair_stats = [], []
for c1, c2 in pairs:
    x, y = rq1[c1].values, rq1[c2].values
    try:
        w, p_raw = wilcoxon(x, y, alternative='two-sided', zero_method='wilcox')
    except ValueError:
        w, p_raw = np.nan, 1.0
    r = rank_biserial(x, y)
    raw_p.append(p_raw)
    pair_stats.append((c1, c2, w, p_raw, r))

adj_p = holm_bonferroni(raw_p)
print(f'\n  {"Comparison":<44} {"W":>6}  {"p raw":>8}  {"p adj":>8}  {"r":>6}')
print('  ' + '-' * 78)
for (c1, c2, w, p_raw, r), p_adj in zip(pair_stats, adj_p):
    label = f'{CONDITION_LABELS[c1]} vs {CONDITION_LABELS[c2]}'
    print(f'  {label:<44} {w:6.1f}  {p_raw:8.4f}  {p_adj:8.4f}  {r:+.3f}  {sig_stars(p_adj)}')

# Directional: C4 vs each baseline 

print('\n' + '═' * 70)
print('  C4 vs EACH BASELINE — One-sided Wilcoxon (H₁: C4 NL < baseline)')
print('═' * 70)

baselines = ['C1_TFIDF_SVM', 'C2a_SciBERT_LR', 'C2b_SciBERT_SVM', 'C3_GAT_only']
raw_p2, rows2 = [], []
for base in baselines:
    x = rq1['C4_GNN_NLP_Fusion'].values
    y = rq1[base].values
    delta = np.mean(x - y)
    try:
        w, p_raw = wilcoxon(x, y, alternative='less', zero_method='wilcox')
    except ValueError:
        w, p_raw = np.nan, 1.0
    r = rank_biserial(x, y)
    raw_p2.append(p_raw)
    rows2.append((base, delta, w, p_raw, r))

adj_p2 = holm_bonferroni(raw_p2)
print(f'\n  {"Baseline":<22}  {"ΔNL (C4−base)":>14}  {"W":>6}  {"p (adj)":>9}  {"r":>6}')
print('  ' + '-' * 66)
for (base, delta, w, p_raw, r), p_adj in zip(rows2, adj_p2):
    print(f'  {CONDITION_LABELS[base]:<22}  {delta:+.4f}          {w:6.1f}  {p_adj:9.4f}  {r:+.3f}  {sig_stars(p_adj)}')

# C2 decomposition: feature effect vs classifier effect 
# Tests the isolation sub-condition: C1→C2b isolates feature change;
# C2b→C2a isolates classifier change.

print('\n' + '═' * 70)
print('  C2 DECOMPOSITION — Feature effect vs Classifier effect')
print('  C1→C2b: same classifier (SVM), richer features')
print('  C2b→C2a: same features (SciBERT), different classifier (LR)')
print('═' * 70)

for label, c_a, c_b in [
    ('Feature effect  (C1 → C2b)', 'C1_TFIDF_SVM',   'C2b_SciBERT_SVM'),
    ('Classifier effect (C2b → C2a)', 'C2b_SciBERT_SVM', 'C2a_SciBERT_LR'),
    ('Graph effect    (C2a → C4)',  'C2a_SciBERT_LR', 'C4_GNN_NLP_Fusion'),
]:
    x = rq1[c_b].values   # "after" condition
    y = rq1[c_a].values   # "before" condition
    delta = np.mean(x - y)
    try:
        w, p = wilcoxon(x, y, alternative='less', zero_method='wilcox')
    except ValueError:
        w, p = np.nan, 1.0
    r = rank_biserial(x, y)
    print(f'\n  {label}')
    print(f'    ΔNL = {delta:+.4f},  W = {w:.1f},  p = {p:.4f}  {sig_stars(p)},  r = {r:+.3f}')

# Win / loss / tie counts 

print('\n' + '═' * 70)
print('  WIN / LOSS / TIE — how often does each condition rank 1st?')
print('═' * 70)
ranks = rq1[CONDITIONS].rank(axis=1, method='min')   # rank 1 = best (lowest NL)
print(f'\n  {"Condition":<22}  {"Rank-1 wins":>12}  {"Mean rank":>10}')
print('  ' + '-' * 48)
for cond in CONDITIONS:
    wins = (ranks[cond] == 1).sum()
    mean_rank = ranks[cond].mean()
    print(f'  {CONDITION_LABELS[cond]:<22}  {wins:>12}  {mean_rank:>10.2f}')

# Per-dataset rank heatmap 

fig, ax = plt.subplots(figsize=(9, 4.5))
rank_matrix = rq1[CONDITIONS].rank(axis=1, method='min').T
cmap = plt.cm.RdYlGn_r   # green = low rank (good), red = high rank

im = ax.imshow(rank_matrix.values, cmap=cmap, vmin=1, vmax=len(CONDITIONS), aspect='auto')

ax.set_xticks(range(len(rq1)))
ax.set_xticklabels([ds.split('_')[0] for ds in rq1.index], rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(len(CONDITIONS)))
ax.set_yticklabels([CONDITION_LABELS[c] for c in CONDITIONS], fontsize=9)

for i in range(len(CONDITIONS)):
    for j in range(len(rq1)):
        val = int(rank_matrix.values[i, j])
        ax.text(j, i, str(val), ha='center', va='center',
                fontsize=9, fontweight='bold',
                color='white' if val >= 4 else 'black')

plt.colorbar(im, ax=ax, label='Rank (1 = best NL)', shrink=0.8)
ax.set_title('RQ1 — Per-dataset condition rank (1 = lowest NL)', fontsize=11, pad=10)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
plt.tight_layout()
plt.savefig('results/rq1_rank_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rq1_rank_heatmap.png')

# Thesis summary table 

print('\n' + '═' * 70)
print('  THESIS TABLE — Mean NL ± Std  across 10 RQ1 datasets')
print('═' * 70)
print(f'\n  {"Condition":<22}  {"Mean NL ± Std":>16}  {"Median":>8}  {"Best rank":>10}')
print('  ' + '-' * 62)
for cond in CONDITIONS:
    v = rq1[cond]
    best = (ranks[cond] == 1).sum()
    print(f'  {CONDITION_LABELS[cond]:<22}  {v.mean():.4f} ± {v.std():.4f}  '
          f'{v.median():8.4f}  {best:>10}×')

In [ ]:
# ── RQ1 VIZ 1: Violin + strip plot ──────────────────────────────────────────
# Shows the full distribution of NL across 10 datasets per condition,
# including spread, IQR, and individual dataset values.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

RESULTS_PATH = Path('results/all_results.csv')
CONDITIONS = ['C1_TFIDF_SVM','C2a_SciBERT_LR','C2b_SciBERT_SVM',
              'C3_GAT_only','C4_GNN_NLP_Fusion']
LABELS     = ['C1\nTF-IDF+SVM','C2a\nSciBERT+LR','C2b\nSciBERT+SVM',
              'C3\nGAT-only','C4\nGNN+NLP']
RQ1_DATASETS = ['Greca_2023','Pijls_2018','Tumkaya_2018','Rinne_2021',
                'Pinos-Cisneros_2023','Abgaz_2023','Erer_2015',
                'Demirkaya_2015','Leenaars_2020','Eggmann_2023']
PALETTE = ['#B0C4DE','#7BA7BC','#A8C5A0','#E6A57E','#534AB7']

all_results = pd.read_csv(RESULTS_PATH)
pivot = (
    all_results.groupby(['dataset','model'])['nl'].mean()
    .reset_index().pivot(index='dataset', columns='model', values='nl')
)
rq1 = pivot.loc[pivot.index.isin(RQ1_DATASETS), CONDITIONS].dropna()

fig, ax = plt.subplots(figsize=(10, 5.5))

positions = np.arange(len(CONDITIONS))
vp = ax.violinplot(
    [rq1[c].values for c in CONDITIONS],
    positions=positions,
    widths=0.6,
    showmedians=False,
    showextrema=False,
)
for i, (body, color) in enumerate(zip(vp['bodies'], PALETTE)):
    body.set_facecolor(color)
    body.set_alpha(0.55)
    body.set_edgecolor('#333333')
    body.set_linewidth(0.8)

# Box inside violin
for i, cond in enumerate(CONDITIONS):
    vals = rq1[cond].values
    q1, med, q3 = np.percentile(vals, [25, 50, 75])
    ax.plot([i, i], [q1, q3], color='#222222', linewidth=3, solid_capstyle='round', zorder=3)
    ax.scatter(i, med, color='white', s=40, zorder=4, edgecolors='#222222', linewidths=1.2)

# Individual dataset points
rng = np.random.default_rng(42)
for i, cond in enumerate(CONDITIONS):
    xs = rng.uniform(-0.12, 0.12, size=len(rq1))
    ax.scatter(positions[i] + xs, rq1[cond].values,
               color=PALETTE[i], s=38, zorder=5,
               edgecolors='#333333', linewidths=0.6, alpha=0.9)

# Pool mean markers
for i, cond in enumerate(CONDITIONS):
    ax.scatter(i, rq1[cond].mean(), marker='D', color='#CC3333',
               s=55, zorder=6, edgecolors='white', linewidths=0.8)

# Annotations: pool mean above each violin
for i, cond in enumerate(CONDITIONS):
    ax.text(i, rq1[cond].max() + 0.025, f'{rq1[cond].mean():.3f}',
            ha='center', va='bottom', fontsize=8.5, color='#333333')

ax.axhline(0.5, color='gray', linewidth=0.9, linestyle=':', alpha=0.6, label='Random baseline (0.5)')
ax.set_xticks(positions)
ax.set_xticklabels(LABELS, fontsize=10)
ax.set_ylabel('Normalised Loss  (↓ better)', fontsize=11)
ax.set_title('RQ1 — NL distribution per condition across 10 datasets', fontsize=12, pad=10)
ax.set_ylim(bottom=max(0, rq1.min().min() - 0.06))
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3, linewidth=0.5)

red_patch = mpatches.Patch(color='#CC3333', label='Pool mean')
ax.legend(handles=[red_patch,
          mpatches.Patch(color='none', label='— median, IQR bar')],
          fontsize=9, framealpha=0.7, loc='upper right')

plt.tight_layout()
plt.savefig('results/rq1_violin_strip.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rq1_violin_strip.png')


In [ ]:
# ── RQ1 VIZ 2: ΔNL lollipop — C4 improvement over C1 per dataset ────────────
# Positive = C4 is better (lower NL). Datasets sorted by improvement.
# Also overlays C2a and C3 deltas to show the full ablation story.

fig, ax = plt.subplots(figsize=(10, 5.5))

deltas = pd.DataFrame({
    'C2a vs C1': rq1['C1_TFIDF_SVM'] - rq1['C2a_SciBERT_LR'],
    'C3 vs C1':  rq1['C1_TFIDF_SVM'] - rq1['C3_GAT_only'],
    'C4 vs C1':  rq1['C1_TFIDF_SVM'] - rq1['C4_GNN_NLP_Fusion'],
}).sort_values('C4 vs C1', ascending=True)

y = np.arange(len(deltas))
labels_ds = [ds.split('_')[0] for ds in deltas.index]

# Stems
for i, (c2a, c3, c4) in enumerate(zip(
        deltas['C2a vs C1'], deltas['C3 vs C1'], deltas['C4 vs C1'])):
    ax.plot([0, c4], [i, i], color='#534AB7', linewidth=1.8, zorder=1, alpha=0.5)

# Dots: C4 (primary), C2a, C3
ax.scatter(deltas['C4 vs C1'], y, color='#534AB7', s=90, zorder=4,
           edgecolors='white', linewidths=0.9, label='C4 GNN+NLP')
ax.scatter(deltas['C2a vs C1'], y, color='#7BA7BC', s=55, zorder=3,
           edgecolors='white', linewidths=0.8, marker='s', label='C2a SciBERT+LR', alpha=0.85)
ax.scatter(deltas['C3 vs C1'], y, color='#E6A57E', s=55, zorder=3,
           edgecolors='white', linewidths=0.8, marker='^', label='C3 GAT-only', alpha=0.85)

ax.axvline(0, color='#333333', linewidth=1.2, linestyle='--', alpha=0.7)
ax.fill_betweenx([-0.5, len(deltas) - 0.5], 0, deltas['C4 vs C1'].max() + 0.02,
                 color='#534AB7', alpha=0.04)

ax.set_yticks(y)
ax.set_yticklabels(labels_ds, fontsize=10)
ax.set_xlabel('NL improvement over C1  (positive = better than baseline)', fontsize=11)
ax.set_title('RQ1 — Per-dataset NL improvement over C1 baseline', fontsize=12, pad=10)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.3, linewidth=0.5)
ax.legend(fontsize=9, framealpha=0.8, loc='lower right')

# Mean lines
for val, color, ls in [
    (deltas['C4 vs C1'].mean(),  '#534AB7', '-'),
    (deltas['C2a vs C1'].mean(), '#7BA7BC', '--'),
    (deltas['C3 vs C1'].mean(),  '#E6A57E', ':'),
]:
    ax.axvline(val, color=color, linewidth=1.2, linestyle=ls, alpha=0.6)
    ax.text(val, len(deltas) - 0.1, f'{val:+.3f}',
            color=color, fontsize=8, ha='center', va='bottom')

plt.tight_layout()
plt.savefig('results/rq1_lollipop_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rq1_lollipop_delta.png')


In [ ]:
# ── RQ1 VIZ 3: NL value heatmap — datasets × conditions ────────────────────
# Colour encodes actual NL value; annotated with value + rank in parentheses.
# Highlights the best condition per dataset with a border.

fig, ax = plt.subplots(figsize=(9, 6))

heat_data = rq1[CONDITIONS].copy()
ranks_mat = heat_data.rank(axis=1, method='min').astype(int)

# Sort datasets by C4 performance
heat_data = heat_data.sort_values('C4_GNN_NLP_Fusion')
ranks_mat = ranks_mat.loc[heat_data.index]

short_names = [ds.split('_')[0] for ds in heat_data.index]
col_labels  = [LABELS[i].replace('\n', ' ') for i in range(len(CONDITIONS))]

im = ax.imshow(heat_data.values, cmap='RdYlGn_r', vmin=0.10, vmax=0.55, aspect='auto')

for i in range(len(heat_data)):
    for j in range(len(CONDITIONS)):
        val  = heat_data.values[i, j]
        rank = ranks_mat.values[i, j]
        text_color = 'white' if val > 0.42 or val < 0.18 else '#222222'
        ax.text(j, i, f'{val:.3f}\n(#{rank})',
                ha='center', va='center', fontsize=8.5,
                color=text_color, fontweight='bold' if rank == 1 else 'normal')
        if rank == 1:
            rect = plt.Rectangle((j - 0.48, i - 0.48), 0.96, 0.96,
                                  fill=False, edgecolor='#222222',
                                  linewidth=2.2, zorder=5)
            ax.add_patch(rect)

ax.set_xticks(range(len(CONDITIONS)))
ax.set_xticklabels(col_labels, fontsize=10, fontweight='bold')
ax.set_yticks(range(len(heat_data)))
ax.set_yticklabels(short_names, fontsize=10)
ax.set_title('RQ1 — Normalised Loss per dataset × condition  (border = best)',
             fontsize=12, pad=10)
ax.tick_params(length=0)
ax.spines[:].set_visible(False)

cbar = plt.colorbar(im, ax=ax, shrink=0.75, pad=0.02)
cbar.set_label('Normalised Loss', fontsize=9)

plt.tight_layout()
plt.savefig('results/rq1_nl_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rq1_nl_heatmap.png')


---
## RQ2: Domain-specific runs

Run all four conditions on the RQ2 domain datasets that are not already covered by the RQ1 LODO folds. Datasets that appear in both RQ1 and RQ2 (Pijls_2018, Tumkaya_2018) reuse their existing LODO results.

In [ ]:
# Identify RQ2-only datasets (not already in RQ1)
rq1_set    = set(rq1_names)
rq2a_names = [d[0] for d in RQ2_DOMAIN_A]
rq2b_names = [d[0] for d in RQ2_DOMAIN_B]

rq2a_only  = [n for n in rq2a_names if n not in rq1_set]
rq2b_only  = [n for n in rq2b_names if n not in rq1_set]

print(f'RQ2 Domain A — additional datasets to run: {rq2a_only}')
print(f'RQ2 Domain B — additional datasets to run: {rq2b_only}')

# Run conditions on RQ2-only datasets
# Using the full domain list as the LODO pool within each domain
if rq2a_only:
    print('\nRunning all conditions on RQ2 Domain A (osteoarthritis)...')
    run_condition1_lodo(rq2a_names, n_runs=N_RUNS)
    run_condition2_lodo(rq2a_names, n_runs=N_RUNS)   # C2a: SciBERT + LR
    run_condition2b_lodo(rq2a_names, n_runs=N_RUNS)  # C2b: SciBERT + SVM (isolation control)
    run_condition3_lodo(rq2a_names, n_runs=N_RUNS)
    run_condition4_lodo(rq2a_names, n_runs=N_RUNS)

if rq2b_only:
    print('\nRunning all conditions on RQ2 Domain B (PTSD/trauma)...')
    run_condition1_lodo(rq2b_names, n_runs=N_RUNS)
    run_condition2_lodo(rq2b_names, n_runs=N_RUNS)   # C2a: SciBERT + LR
    run_condition2b_lodo(rq2b_names, n_runs=N_RUNS)  # C2b: SciBERT + SVM (isolation control)
    run_condition3_lodo(rq2b_names, n_runs=N_RUNS)
    run_condition4_lodo(rq2b_names, n_runs=N_RUNS)

print('\nRQ2 runs complete.')

In [ ]:
def compute_graph_densities() -> dict:
    """
    Compute citation graph density for each dataset.

    Density = number of internal edges / (N * N).
    Higher density means more of the potential citation edges actually exist.
    """
    densities = {}
    for name, df in DATASETS.items():
        edge_index, stats = build_citation_graph(df)
        densities[name] = stats['density']
    return densities


def plot_rq2_density_scatter(
    domain_datasets: list[tuple],
    domain_name: str,
    results_path: Path = RESULTS_PATH,
) -> None:
    """
    Scatter plot: graph density (x-axis) vs Normalised Loss improvement over C1 (y-axis)
    for datasets in one RQ2 domain.

    'Improvement' = NL(C1) - NL(C4). Positive = fusion beat the baseline.
    Negative = fusion was worse than baseline for this dataset.
    """
    df_results = pd.read_csv(results_path)
    densities  = compute_graph_densities()

    names = [d[0] for d in domain_datasets]

    points = []
    for name in names:
        c1_nl = df_results[(df_results['dataset'] == name) &
                            (df_results['model'] == 'C1_TFIDF_SVM')]['nl'].mean()
        c4_nl = df_results[(df_results['dataset'] == name) &
                            (df_results['model'] == 'C4_GNN_NLP_Fusion')]['nl'].mean()

        if np.isnan(c1_nl) or np.isnan(c4_nl):
            continue

        points.append({
            'name':       name,
            'density':    densities.get(name, 0),
            'improvement': c1_nl - c4_nl,  # positive = fusion better
        })

    pts = pd.DataFrame(points)
    if len(pts) == 0:
        print(f'No data for {domain_name} — run experiments first.')
        return

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(pts['density'], pts['improvement'], s=80, color='#534AB7', alpha=0.8, zorder=3)

    # Label each point with the dataset name (short)
    for _, row in pts.iterrows():
        ax.annotate(
            row['name'].split('_')[0],  # just the surname
            xy=(row['density'], row['improvement']),
            xytext=(4, 4), textcoords='offset points',
            fontsize=8,
        )

    # Add trend line
    if len(pts) >= 3:
        z = np.polyfit(pts['density'], pts['improvement'], 1)
        p = np.poly1d(z)
        xs = np.linspace(pts['density'].min(), pts['density'].max(), 100)
        ax.plot(xs, p(xs), '--', color='#D85A30', linewidth=1, alpha=0.7, label='trend')

    ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.6)
    ax.set_xlabel('Citation graph density', fontsize=10)
    ax.set_ylabel('NL improvement over C1 (higher = fusion better)', fontsize=10)
    ax.set_title(f'RQ2 — {domain_name}: graph density vs fusion improvement', fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(alpha=0.3, linewidth=0.5)

    plt.tight_layout()
    fname = f'results/rq2_{domain_name.lower().replace("/","_").replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')


# Run both RQ2 domain plots
plot_rq2_density_scatter(RQ2_DOMAIN_A, 'Osteoarthritis')
plot_rq2_density_scatter(RQ2_DOMAIN_B, 'PTSD Trauma')

In [ ]:
# ─── RQ2 Statistical Analysis ────────────────────────────────────────────────
# Compares all four ablation conditions within each domain (osteoarthritis,
# PTSD/trauma) using non-parametric tests appropriate for N=6 datasets.
#
# Unit of analysis: dataset-level mean NL (averaged across N_RUNS seeds).
# This avoids pseudoreplication from repeated random seeds.
#
# Tests:
#   1. Descriptive table (mean ± std per condition × domain)
#   2. Friedman test — any condition effect within each domain?
#   3. Pairwise Wilcoxon signed-rank tests (C4 vs each baseline, all pairs)
#      with Bonferroni–Holm correction
#   4. Rank-biserial correlation effect sizes
#   5. Spearman correlation — graph density vs C4 improvement
#   6. Mann-Whitney U — does C4 improvement differ between the two domains?

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from scipy import stats
from scipy.stats import (
    friedmanchisquare, wilcoxon, mannwhitneyu, spearmanr,
)
from itertools import combinations

RESULTS_PATH = Path('results/all_results.csv')

CONDITIONS = ['C1_TFIDF_SVM', 'C2a_SciBERT_LR', 'C2b_SciBERT_SVM',
              'C3_GAT_only', 'C4_GNN_NLP_Fusion']
CONDITION_LABELS = {
    'C1_TFIDF_SVM':       'C1 TF-IDF+SVM',
    'C2a_SciBERT_LR':     'C2a SciBERT+LR',
    'C2b_SciBERT_SVM':    'C2b SciBERT+SVM',
    'C3_GAT_only':        'C3 GAT-only',
    'C4_GNN_NLP_Fusion':  'C4 GNN+NLP',
}

RQ2_DOMAIN_A = [
    'Pijls_2018', 'Cozim-Melges_2024', 'Pinos-Cisneros_2023',
    'van_der_Valk_2021', 'Meijboom_2021', 'Tektonidou_2019',
]
RQ2_DOMAIN_B = [
    'Tumkaya_2018', 'Rinne_2021', 'Donners_2021',
    'Smid_2019', 'Boersma-van_Dam_2024', 'Oud_2018',
]

DOMAINS = {
    'Osteoarthritis': RQ2_DOMAIN_A,
    'PTSD / Trauma':  RQ2_DOMAIN_B,
}

# Helper: rank-biserial correlation for Wilcoxon

def rank_biserial(x, y):
    """Effect size r for a paired Wilcoxon test (matched differences)."""
    d = np.array(x) - np.array(y)
    d = d[d != 0]
    if len(d) == 0:
        return np.nan
    ranks = stats.rankdata(np.abs(d))
    r_plus  = np.sum(ranks[d > 0])
    r_minus = np.sum(ranks[d < 0])
    n = len(d)
    return (r_plus - r_minus) / (n * (n + 1) / 2)

# Helper: Holm–Bonferroni correction 

def holm_bonferroni(p_values: list[float]) -> list[float]:
    """Return Holm-corrected p-values (same order as input)."""
    n = len(p_values)
    order = np.argsort(p_values)
    corrected = np.zeros(n)
    for rank, idx in enumerate(order):
        corrected[idx] = min(p_values[idx] * (n - rank), 1.0)
    # enforce monotonicity
    for i in range(1, n):
        corrected[order[i]] = max(corrected[order[i]], corrected[order[i - 1]])
    return corrected.tolist()

# Load and pivot to dataset-level means 

all_results = pd.read_csv(RESULTS_PATH)

# Mean NL per (dataset × model), averaged across seeds
pivot = (
    all_results
    .groupby(['dataset', 'model'])['nl']
    .mean()
    .reset_index()
    .pivot(index='dataset', columns='model', values='nl')
)

print('Datasets in results CSV:')
print(sorted(all_results['dataset'].unique()))

# Descriptive statistics 

print('\n' + '═' * 72)
print('  RQ2 DESCRIPTIVE STATISTICS — Mean Normalised Loss (↓ better)')
print('═' * 72)

for domain_name, datasets in DOMAINS.items():
    sub = pivot.loc[pivot.index.isin(datasets), CONDITIONS].dropna(how='all')
    print(f'\n  Domain: {domain_name}  (N = {len(sub)} datasets)')
    print(f'  {"Condition":<22} {"Mean NL":>8}  {"Std":>7}  {"Min":>7}  {"Max":>7}')
    print('  ' + '-' * 58)
    for cond in CONDITIONS:
        if cond not in sub.columns:
            print(f'  {CONDITION_LABELS[cond]:<22}  (no data)')
            continue
        vals = sub[cond].dropna()
        print(f'  {CONDITION_LABELS[cond]:<22}  {vals.mean():7.4f}  '
              f'{vals.std():7.4f}  {vals.min():7.4f}  {vals.max():7.4f}')

# Friedman test — any overall condition effect? 

print('\n' + '═' * 72)
print('  FRIEDMAN TEST — H₀: all conditions equal within each domain')
print('═' * 72)

for domain_name, datasets in DOMAINS.items():
    sub = pivot.loc[pivot.index.isin(datasets), CONDITIONS].dropna()
    if len(sub) < 3:
        print(f'\n  {domain_name}: insufficient data (N={len(sub)})')
        continue
    groups = [sub[c].values for c in CONDITIONS if c in sub.columns]
    stat, p = friedmanchisquare(*groups)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'\n  {domain_name}: χ²({len(groups)-1}) = {stat:.3f},  p = {p:.4f}  {sig}')

# Pairwise Wilcoxon signed-rank tests with Holm correction 

print('\n' + '═' * 72)
print('  PAIRWISE WILCOXON SIGNED-RANK TESTS (Holm–Bonferroni corrected)')
print('  Effect size: rank-biserial r  (|r| > 0.5 = large)')
print('═' * 72)

for domain_name, datasets in DOMAINS.items():
    sub = pivot.loc[pivot.index.isin(datasets), CONDITIONS].dropna()
    print(f'\n  Domain: {domain_name}  (N = {len(sub)})')

    pairs = list(combinations(CONDITIONS, 2))
    raw_p = []
    pair_data = []

    for c1, c2 in pairs:
        if c1 not in sub.columns or c2 not in sub.columns:
            continue
        x, y = sub[c1].values, sub[c2].values
        if len(x) < 3:
            continue
        try:
            stat, p = wilcoxon(x, y, alternative='two-sided', zero_method='wilcox')
        except ValueError:
            stat, p = np.nan, 1.0
        r = rank_biserial(x, y)
        raw_p.append(p)
        pair_data.append((c1, c2, stat, p, r))

    corrected = holm_bonferroni(raw_p)
    print(f'  {"Comparison":<42} {"W":>6}  {"p (raw)":>8}  {"p (adj)":>8}  {"r":>6}')
    print('  ' + '-' * 76)
    for (c1, c2, stat, p_raw, r), p_adj in zip(pair_data, corrected):
        sig = '***' if p_adj < 0.001 else ('**' if p_adj < 0.01 else ('*' if p_adj < 0.05 else 'ns'))
        label = f'{CONDITION_LABELS[c1]} vs {CONDITION_LABELS[c2]}'
        print(f'  {label:<42} {stat:6.1f}  {p_raw:8.4f}  {p_adj:8.4f}  {r:+.3f}  {sig}')

# Focused comparison: C4 vs each other condition 

print('\n' + '═' * 72)
print('  C4 (GNN+NLP Fusion) vs Each Baseline — Directional (one-sided)')
print('  H₁: C4 NL < baseline NL  (C4 is better, lower NL)')
print('═' * 72)

baselines = ['C1_TFIDF_SVM', 'C2a_SciBERT_LR', 'C2b_SciBERT_SVM', 'C3_GAT_only']

for domain_name, datasets in DOMAINS.items():
    sub = pivot.loc[pivot.index.isin(datasets), CONDITIONS].dropna()
    print(f'\n  Domain: {domain_name}  (N = {len(sub)})')
    print(f'  {"Baseline":<22} {"ΔNL (C4−base)":>14}  {"W":>6}  {"p (one-sided)":>14}  {"r":>6}')
    print('  ' + '-' * 70)

    raw_p = []
    rows = []
    for base in baselines:
        if base not in sub.columns or 'C4_GNN_NLP_Fusion' not in sub.columns:
            continue
        x = sub['C4_GNN_NLP_Fusion'].values
        y = sub[base].values
        delta = np.mean(x - y)
        try:
            stat, p = wilcoxon(x, y, alternative='less', zero_method='wilcox')
        except ValueError:
            stat, p = np.nan, 1.0
        r = rank_biserial(x, y)
        raw_p.append(p)
        rows.append((base, delta, stat, p, r))

    corrected = holm_bonferroni(raw_p)
    for (base, delta, stat, p_raw, r), p_adj in zip(rows, corrected):
        sig = '***' if p_adj < 0.001 else ('**' if p_adj < 0.01 else ('*' if p_adj < 0.05 else 'ns'))
        print(f'  {CONDITION_LABELS[base]:<22}  {delta:+.4f}        {stat:6.1f}  {p_adj:14.4f}  {r:+.3f}  {sig}')

# Spearman: graph density vs C4 improvement 

print('\n' + '═' * 72)
print('  SPEARMAN CORRELATION — Citation Graph Density vs C4 Improvement')
print('  Improvement = NL(C1) − NL(C4)  (positive = fusion beats baseline)')
print('═' * 72)

try:
    densities = compute_graph_densities()   # defined in Cell 55; assumes DATASETS is loaded
    for domain_name, datasets in DOMAINS.items():
        sub = pivot.loc[pivot.index.isin(datasets), ['C1_TFIDF_SVM', 'C4_GNN_NLP_Fusion']].dropna()
        rho_data = []
        for ds in sub.index:
            if ds not in densities:
                continue
            improvement = sub.loc[ds, 'C1_TFIDF_SVM'] - sub.loc[ds, 'C4_GNN_NLP_Fusion']
            rho_data.append({'dataset': ds, 'density': densities[ds], 'improvement': improvement})
        if len(rho_data) < 3:
            print(f'\n  {domain_name}: insufficient data for correlation')
            continue
        df_rho = pd.DataFrame(rho_data)
        rho, p = spearmanr(df_rho['density'], df_rho['improvement'])
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f'\n  {domain_name}: ρ = {rho:+.3f},  p = {p:.4f}  {sig}  (N = {len(df_rho)})')
except NameError:
    print('\n  [Skipped] compute_graph_densities() not available — run Cell 55 first.')

# Cross-domain: does fusion improve more in one domain? 

print('\n' + '═' * 72)
print('  CROSS-DOMAIN COMPARISON — Mann-Whitney U')
print('  H₀: C4 improvement (NL_C1 − NL_C4) is the same in both domains')
print('═' * 72)

def domain_improvements(datasets):
    sub = pivot.loc[pivot.index.isin(datasets), ['C1_TFIDF_SVM', 'C4_GNN_NLP_Fusion']].dropna()
    return (sub['C1_TFIDF_SVM'] - sub['C4_GNN_NLP_Fusion']).values

imp_a = domain_improvements(RQ2_DOMAIN_A)
imp_b = domain_improvements(RQ2_DOMAIN_B)

if len(imp_a) >= 3 and len(imp_b) >= 3:
    stat, p = mannwhitneyu(imp_a, imp_b, alternative='two-sided')
    print(f'\n  Osteoarthritis improvements: {imp_a}')
    print(f'  PTSD/Trauma improvements:    {imp_b}')
    print(f'\n  Mean improvement — Osteoarthritis: {imp_a.mean():+.4f}')
    print(f'  Mean improvement — PTSD/Trauma:    {imp_b.mean():+.4f}')
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'\n  Mann-Whitney U = {stat:.1f},  p = {p:.4f}  {sig}')
else:
    print('\n  Insufficient data for cross-domain comparison.')

# Summary table for thesis 

print('\n' + '═' * 72)
print('  SUMMARY: Pool mean NL per condition × domain')
print('═' * 72)
print(f'\n  {"Condition":<22}  {"Osteoarthritis":>16}  {"PTSD/Trauma":>14}')
print('  ' + '-' * 58)
for cond in CONDITIONS:
    vals_a = pivot.loc[pivot.index.isin(RQ2_DOMAIN_A), cond].dropna()
    vals_b = pivot.loc[pivot.index.isin(RQ2_DOMAIN_B), cond].dropna()
    str_a = f'{vals_a.mean():.4f} ± {vals_a.std():.4f}' if len(vals_a) else '—'
    str_b = f'{vals_b.mean():.4f} ± {vals_b.std():.4f}' if len(vals_b) else '—'
    print(f'  {CONDITION_LABELS[cond]:<22}  {str_a:>16}  {str_b:>14}')

In [ ]:
# ── RQ2 VIZ 1: Grouped bar chart — conditions × domain (FIXED) ───────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

RESULTS_PATH = Path('results/all_results.csv')
CONDITIONS = ['C1_TFIDF_SVM','C2a_SciBERT_LR','C2b_SciBERT_SVM',
              'C3_GAT_only','C4_GNN_NLP_Fusion']
LABELS     = ['C1\nTF-IDF+SVM','C2a\nSciBERT+LR','C2b\nSciBERT+SVM',
              'C3\nGAT-only','C4\nGNN+NLP']

RQ2_DOMAIN_A = ['Pijls_2018','Cozim-Melges_2024','Pinos-Cisneros_2023',
                 'van_der_Valk_2021','Meijboom_2021','Tektonidou_2019']
RQ2_DOMAIN_B = ['Tumkaya_2018','Rinne_2021','Donners_2021',
                 'Smid_2019','Boersma-van_Dam_2024','Oud_2018']
DOMAINS    = {'Osteoarthritis': RQ2_DOMAIN_A, 'PTSD / Trauma': RQ2_DOMAIN_B}
DOM_COLORS = {'Osteoarthritis': '#5B8DB8', 'PTSD / Trauma': '#D4845A'}

all_results = pd.read_csv(RESULTS_PATH)
pivot = (
    all_results.groupby(['dataset','model'])['nl'].mean()
    .reset_index().pivot(index='dataset', columns='model', values='nl')
)

fig, ax = plt.subplots(figsize=(11, 5.5))

n_dom = len(DOMAINS)
width = 0.32
gap   = 0.08

for di, (dom_name, datasets) in enumerate(DOMAINS.items()):
    sub   = pivot.loc[pivot.index.isin(datasets), CONDITIONS].dropna()
    means = sub.mean()
    stds  = sub.std()
    offset = (di - (n_dom - 1) / 2) * (width + gap / n_dom)

    bars = ax.bar(
        np.arange(len(CONDITIONS)) + offset,
        means[CONDITIONS],
        width=width,
        color=DOM_COLORS[dom_name],
        alpha=0.85,
        label=f'{dom_name}  (N={len(sub)})',
        edgecolor='white',
        linewidth=0.5,
    )
    ax.errorbar(
        np.arange(len(CONDITIONS)) + offset,
        means[CONDITIONS],
        yerr=stds[CONDITIONS],
        fmt='none', color='#333333', capsize=4, linewidth=1.2, zorder=5,
    )
    # FIX: iterate over (bar, condition) pairs directly
    for bar, cond in zip(bars, CONDITIONS):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            means[cond] + stds[cond] + 0.012,
            f'{means[cond]:.3f}',
            ha='center', va='bottom', fontsize=7.5, color='#333333',
        )

ax.axhline(0.5, color='gray', linewidth=0.9, linestyle=':', alpha=0.5, label='Random (0.5)')
ax.set_xticks(np.arange(len(CONDITIONS)))
ax.set_xticklabels(LABELS, fontsize=10)
ax.set_ylabel('Mean Normalised Loss  (↓ better)', fontsize=11)
ax.set_title('RQ2 — Mean NL per condition across domains  (error bars = ±1 SD)', fontsize=12, pad=10)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3, linewidth=0.5)
ax.legend(fontsize=10, framealpha=0.8)
ax.set_ylim(0, 0.62)

plt.tight_layout()
plt.savefig('results/rq2_grouped_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rq2_grouped_bar.png')


In [ ]:
# ── RQ2 VIZ 2: Slope chart — condition ranks in Domain A vs Domain B ─────────
# Shows which conditions maintain / swap rank when moving between domains.
# Crossing lines = rank inversion between the two domains.

fig, ax = plt.subplots(figsize=(7, 5))

dom_means = {}
for dom_name, datasets in DOMAINS.items():
    sub = pivot.loc[pivot.index.isin(datasets), CONDITIONS].dropna()
    dom_means[dom_name] = sub.mean()

dom_names = list(dom_means.keys())
ranks = {}
for dom in dom_names:
    ranked = dom_means[dom][CONDITIONS].rank().astype(int)
    ranks[dom] = ranked

COND_COLORS = dict(zip(CONDITIONS, ['#B0C4DE','#7BA7BC','#A8C5A0','#E6A57E','#534AB7']))

for cond in CONDITIONS:
    r_a = ranks[dom_names[0]][cond]
    r_b = ranks[dom_names[1]][cond]
    ax.plot([0, 1], [r_a, r_b],
            color=COND_COLORS[cond], linewidth=2.4,
            marker='o', markersize=9,
            markerfacecolor=COND_COLORS[cond],
            markeredgecolor='white', markeredgewidth=1.2,
            label=CONDITION_LABELS[cond], zorder=3)
    # Labels on both sides
    ax.text(-0.06, r_a, f'#{r_a}  {CONDITION_LABELS[cond]}',
            ha='right', va='center', fontsize=9, color=COND_COLORS[cond])
    ax.text(1.06, r_b, f'#{r_b}  {CONDITION_LABELS[cond]}',
            ha='left',  va='center', fontsize=9, color=COND_COLORS[cond])

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0.5, len(CONDITIONS) + 0.5)
ax.invert_yaxis()  # rank 1 at the top
ax.set_xticks([0, 1])
ax.set_xticklabels(dom_names, fontsize=12, fontweight='bold')
ax.set_ylabel('Rank  (1 = lowest NL)', fontsize=11)
ax.set_title('RQ2 — Condition rank by domain\n(crossing = rank swap between domains)',
             fontsize=12, pad=8)
ax.yaxis.set_visible(False)
ax.spines[['top','right','left']].set_visible(False)
ax.tick_params(bottom=False)

# Mean NL annotations below each domain column
for xi, dom in enumerate(dom_names):
    for cond in CONDITIONS:
        ax.annotate(
            f'{dom_means[dom][cond]:.3f}',
            xy=(xi, ranks[dom][cond]),
            xytext=(0, -14), textcoords='offset points',
            ha='center', fontsize=7.5, color='#555555',
        )

plt.tight_layout()
plt.savefig('results/rq2_slope_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rq2_slope_chart.png')


In [ ]:
# ── RQ2 VIZ 3: Per-dataset strip plot, faceted by domain ────────────────────
# Each dot = one dataset's mean NL. Connected dots show how the same
# dataset's score moves across conditions. Reveals outliers and consistency.

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)

for ax, (dom_name, datasets) in zip(axes, DOMAINS.items()):
    sub = pivot.loc[pivot.index.isin(datasets), CONDITIONS].dropna()
    short = [ds.split('_')[0] for ds in sub.index]

    ds_colors = plt.cm.tab10(np.linspace(0, 0.85, len(sub)))

    # Draw connecting lines first (one per dataset)
    for i, (ds, row) in enumerate(sub.iterrows()):
        ax.plot(range(len(CONDITIONS)), row[CONDITIONS].values,
                color=ds_colors[i], linewidth=1.1, alpha=0.45, zorder=1)

    # Dots on top
    for j, cond in enumerate(CONDITIONS):
        for i, (ds, row) in enumerate(sub.iterrows()):
            ax.scatter(j, row[cond],
                       color=ds_colors[i], s=60, zorder=3,
                       edgecolors='white', linewidths=0.8, alpha=0.95)

    # Domain mean line
    means = sub[CONDITIONS].mean()
    ax.plot(range(len(CONDITIONS)), means.values,
            color='#222222', linewidth=2.2, linestyle='--',
            marker='D', markersize=7, zorder=4,
            markerfacecolor='#CC3333', markeredgecolor='white',
            label='Domain mean')

    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels(LABELS, fontsize=9)
    ax.set_title(f'{dom_name}  (N={len(sub)})', fontsize=12, fontweight='bold', pad=8)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.25, linewidth=0.5)

    # Custom legend for dataset names
    patches = [mpatches.Patch(color=ds_colors[i], label=short[i])
               for i in range(len(sub))]
    patches.append(mpatches.Patch(color='#CC3333', label='Domain mean'))
    ax.legend(handles=patches, fontsize=8, framealpha=0.75,
              loc='upper right', ncol=1)

axes[0].set_ylabel('Mean Normalised Loss  (↓ better)', fontsize=11)
fig.suptitle('RQ2 — Per-dataset NL across conditions, by domain', fontsize=13, y=1.01)

plt.tight_layout()
plt.savefig('results/rq2_strip_by_domain.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/rq2_strip_by_domain.png')


---
## Final results snapshot

This cell prints the complete state of `results/all_results.csv` for a sanity check before the experiment freeze.

In [ ]:
final = load_results()
print(f'Total rows in results CSV: {len(final)}')
print(f'Models logged: {sorted(final["model"].unique())}')
print(f'Datasets logged: {sorted(final["dataset"].unique())}')
print()
print('Mean Normalised Loss by condition (across all datasets and runs):')
print(
    final.groupby('model')['nl']
    .agg(['mean', 'std', 'count'])
    .rename(columns={'mean': 'NL_mean', 'std': 'NL_std', 'count': 'n_runs'})
    .round(4)
    .sort_values('NL_mean')
    .to_string()
)
print()
print('Runtime summary (seconds per run):')
print(
    final.groupby('model')['runtime_s']
    .agg(['mean', 'max'])
    .round(1)
    .to_string()
)